In [1]:
import pandas as pd
path = "https://raw.githubusercontent.com/ulka29/paper/refs/heads/main/01_data/processed/engineered_feature_dataset.csv"# import Check dataset
df= pd.read_csv(path)
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound,word_count,...,second_person_count,exclamation_count,question_count,comma_count,period_count,repeated_exclamation_count,repeated_question_count,repeated_punctuation_count,mention_count,hashtag_count
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,9.0,...,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,2.0
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000,14.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137,9.0,...,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258,18.0,...,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,1.0,0.0


In [2]:

# ============================================================
# CELL 1 — SETUP + DATA PREPARATION (uses your existing df)
# ============================================================

import time
import warnings
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from IPython.display import display
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.base import clone

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
N_SPLITS = 5
RANDOM_STATE = 42

TEXT_COL = "lemmatized_tweet"
RAW_TEXT_COL = "tweet_text"
TARGET_COL = "cyberbullying_type"

NUMERIC_FEATURES = [
    "negative", "positive", "neutral", "compound",
    "second_person_count", "mention_count", "hashtag_count",
    "profanity_count", "unique_profanity_count",
    "profanity_ratio", "profanity_presence"
]

FOCUS_CLASSES = [
    "not_cyberbullying",
    "other_cyberbullying"
]

# TF-IDF settings: sublinear_tf=False is the default
TFIDF_ANALYZER = "word"
TFIDF_NGRAM_RANGE = (1, 2)
TFIDF_MAX_FEATURES = 10000
TFIDF_SUBLINEAR_TF = False

# Neural-model setting
MLP_SVD_COMPONENTS = 300

# ---------------- DATA ----------------
required_cols = (
    [RAW_TEXT_COL, TEXT_COL] + NUMERIC_FEATURES + [TARGET_COL]
)
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df_cv = df[required_cols].copy()

# Preserve original df index for downstream joins/error sheets
df_cv["row_id"] = df_cv.index

df_cv = df_cv.dropna(subset=[TARGET_COL]).copy()
df_cv = df_cv[
    df_cv[TARGET_COL].astype(str).str.strip().ne("")
].copy()

X_text = df_cv[TEXT_COL].fillna("").astype(str).to_numpy()

X_num = (
    df_cv[NUMERIC_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .to_numpy(dtype=np.float32)
)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(
    df_cv[TARGET_COL].astype(str)
)

class_names = [str(c) for c in label_encoder.classes_]
class_ids = np.arange(len(class_names))
class_to_id = {c: i for i, c in enumerate(class_names)}

print("Total rows used for CV :", len(y))
print("Classes                :", class_names)
print("Class counts:")
print(pd.Series(y).map(
    dict(enumerate(class_names))
).value_counts())

print(
    "Missing text rows      :",
    int(df_cv[TEXT_COL].isna().sum()),
    "(filled with empty string)"
)
print("Numeric features       :", len(NUMERIC_FEATURES))
print("Class weighting        : None")
print("TF-IDF sublinear_tf    :", TFIDF_SUBLINEAR_TF)


Total rows used for CV : 42602
Classes                : ['age', 'ethnicity', 'gender', 'not_cyberbullying', 'other_cyberbullying', 'religion']
Class counts:
religion               7963
age                    7959
gender                 7478
ethnicity              7426
not_cyberbullying      6435
other_cyberbullying    5341
Name: count, dtype: int64
Missing text rows      : 0 (filled with empty string)
Numeric features       : 11
Class weighting        : None
TF-IDF sublinear_tf    : False


In [3]:

# ============================================================
# CELL 2 — BUILD STRATIFIED K-FOLD FEATURE MATRICES
# TF-IDF / scaling / SVD fitted on each fold's training part only
# ============================================================

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

fold_data = []

for fold, (tr_idx, va_idx) in enumerate(
    tqdm(
        skf.split(X_text, y),
        total=N_SPLITS,
        desc="Building folds"
    ),
    start=1
):
    # 1. TF-IDF: fit on training text only
    tfidf = TfidfVectorizer(
        analyzer=TFIDF_ANALYZER,
        ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES,
        sublinear_tf=TFIDF_SUBLINEAR_TF,
        dtype=np.float32
    )

    W_tr = tfidf.fit_transform(X_text[tr_idx])
    W_va = tfidf.transform(X_text[va_idx])

    # 2. Scale numeric features using training fold only
    scaler = StandardScaler()

    N_tr_scaled = scaler.fit_transform(
        X_num[tr_idx]
    ).astype(np.float32)

    N_va_scaled = scaler.transform(
        X_num[va_idx]
    ).astype(np.float32)

    # 3. Linear models: sparse TF-IDF + scaled numeric features
    X_tr_linear = hstack(
        [W_tr, csr_matrix(N_tr_scaled)],
        format="csr",
        dtype=np.float32
    )

    X_va_linear = hstack(
        [W_va, csr_matrix(N_va_scaled)],
        format="csr",
        dtype=np.float32
    )

    # 4. Tree models: same features, numeric features unscaled
    X_tr_tree = hstack(
        [W_tr, csr_matrix(X_num[tr_idx])],
        format="csr",
        dtype=np.float32
    )

    X_va_tree = hstack(
        [W_va, csr_matrix(X_num[va_idx])],
        format="csr",
        dtype=np.float32
    )

    # 5. MLP: training-fold-only SVD + scaled numeric features
    n_components = min(
        MLP_SVD_COMPONENTS,
        W_tr.shape[1] - 1,
        W_tr.shape[0] - 1
    )

    if n_components < 1:
        raise ValueError("Not enough TF-IDF dimensions for SVD.")

    svd = TruncatedSVD(
        n_components=n_components,
        random_state=RANDOM_STATE
    )

    Z_tr = svd.fit_transform(W_tr).astype(np.float32)
    Z_va = svd.transform(W_va).astype(np.float32)

    X_tr_mlp = np.hstack([Z_tr, N_tr_scaled]).astype(np.float32)
    X_va_mlp = np.hstack([Z_va, N_va_scaled]).astype(np.float32)

    fold_data.append({
        "fold": fold,
        "tr_idx": tr_idx,
        "va_idx": va_idx,
        "row_ids": df_cv["row_id"].to_numpy()[va_idx],
        "y_tr": y[tr_idx],
        "y_va": y[va_idx],

        # Backward-compatible keys: unscaled numeric/tree matrix
        "X_tr": X_tr_tree,
        "X_va": X_va_tree,

        # Explicit model-specific matrices
        "X_tr_linear": X_tr_linear,
        "X_va_linear": X_va_linear,
        "X_tr_tree": X_tr_tree,
        "X_va_tree": X_va_tree,
        "X_tr_mlp": X_tr_mlp,
        "X_va_mlp": X_va_mlp,

        # Fold-specific preprocessing objects, if needed later
        "tfidf": tfidf,
        "scaler": scaler,
        "svd": svd
    })

    print(
        f"Fold {fold}: train={len(tr_idx)}, "
        f"validation={len(va_idx)}, "
        f"TF-IDF={W_tr.shape[1]}, "
        f"linear/tree features={X_tr_tree.shape[1]}, "
        f"MLP features={X_tr_mlp.shape[1]}"
    )

print(f"\nBuilt {len(fold_data)} folds.")


Building folds:   0%|          | 0/5 [00:00<?, ?it/s]

Fold 1: train=34081, validation=8521, TF-IDF=10000, linear/tree features=10011, MLP features=311
Fold 2: train=34081, validation=8521, TF-IDF=10000, linear/tree features=10011, MLP features=311
Fold 3: train=34082, validation=8520, TF-IDF=10000, linear/tree features=10011, MLP features=311
Fold 4: train=34082, validation=8520, TF-IDF=10000, linear/tree features=10011, MLP features=311
Fold 5: train=34082, validation=8520, TF-IDF=10000, linear/tree features=10011, MLP features=311

Built 5 folds.


In [5]:

# ============================================================
# CELL 3 — MODEL DEFINITIONS
# Six primary models + optional MLP
# Best parameters applied; no class weighting
# ============================================================

# XGB_DEVICE = "cpu"
XGB_DEVICE = "cuda"

models = {
    "Logistic Regression": LogisticRegression(
        C=1.0,
        class_weight=None,
        max_iter=10000,
        solver="lbfgs",
        random_state=RANDOM_STATE
    ),

    "Linear SVM": LinearSVC(
        C=0.1,
        class_weight=None,
        max_iter=10000,
        random_state=RANDOM_STATE
    ),

    "SGD Classifier": SGDClassifier(
        alpha=1e-5,
        class_weight=None,
        penalty="l2",
        loss="hinge",
        max_iter=10000,
        tol=1e-3,
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        class_weight=None,
        max_depth=None,
        max_features="sqrt",
        min_samples_leaf=1,
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        class_weight=None,
        max_depth=None,
        max_features="sqrt",
        min_samples_leaf=1,
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        num_class=len(class_names),
        eval_metric="mlogloss",
        tree_method="hist",
        device=XGB_DEVICE,
        random_state=RANDOM_STATE,
        n_jobs=1,
        colsample_bytree=0.8,
        learning_rate=0.1,
        max_depth=8,
        n_estimators=400,
        subsample=0.8
    )
}

# Optional neural benchmark
# sklearn MLP runs on CPU and does not use CUDA.
models["MLP Neural Network"] = MLPClassifier(
    solver="adam",
    activation="relu",
    hidden_layer_sizes=(128,),
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=8,
    random_state=RANDOM_STATE
)

model_names = list(models.keys())

linear_models = {
    "Logistic Regression",
    "Linear SVM",
    "SGD Classifier"
}

tree_models = {
    "Random Forest",
    "Extra Trees",
    "XGBoost"
}

neural_models = {
    "MLP Neural Network"
}

print("Primary models:", model_names[:6])
print("Optional neural model:", model_names[6:])
print("Class weighting: None for all models")
print("XGBoost device:", XGB_DEVICE)

# Display final model configurations
for name, model in models.items():
    print(f"\n{name}:")
    print(model.get_params())


Primary models: ['Logistic Regression', 'Linear SVM', 'SGD Classifier', 'Random Forest', 'Extra Trees', 'XGBoost']
Optional neural model: ['MLP Neural Network']
Class weighting: None for all models
XGBoost device: cuda

Logistic Regression:
{'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 10000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}

Linear SVM:
{'C': 0.1, 'class_weight': None, 'dual': 'auto', 'fit_intercept': True, 'intercept_scaling': 1, 'loss': 'squared_hinge', 'max_iter': 10000, 'multi_class': 'ovr', 'penalty': 'l2', 'random_state': 42, 'tol': 0.0001, 'verbose': 0}

SGD Classifier:
{'alpha': 1e-05, 'average': False, 'class_weight': None, 'early_stopping': False, 'epsilon': 0.1, 'eta0': 0.0, 'fit_intercept': True, 'l1_ratio': 0.15, 'learning_rate': 'optimal', 'loss': 'hinge', 'max_iter': 10000, 'n

In [6]:

# ============================================================
# CELL 4 — K-FOLD OOF PREDICTIONS
# One out-of-fold prediction per row per model
# ============================================================

# IMPORTANT:
# If preprocessing/model settings changed, clear old cached results:
# oof_pred = {}
# fold_macro_f1 = {}
# run_minutes = {}
# oof_fold_id = np.full(len(y), -1, dtype=int)
# oof_results = pd.DataFrame()

if "oof_pred" not in globals():
    oof_pred = {}

if "fold_macro_f1" not in globals():
    fold_macro_f1 = {}

if "run_minutes" not in globals():
    run_minutes = {}

if "oof_fold_id" not in globals():
    oof_fold_id = np.full(len(y), -1, dtype=int)

for fd in fold_data:
    oof_fold_id[fd["va_idx"]] = fd["fold"]

for name, template in models.items():

    if name in oof_pred:
        print(f"[skip] {name} already has cached OOF predictions")
        continue

    preds = np.full(len(y), -1, dtype=int)
    fold_scores = []
    t0 = time.time()

    print(f"\nStarting: {name}")

    for fd in tqdm(fold_data, desc=name, leave=True):

        model = clone(template)

        # Choose the correct feature matrix for this model family
        if name in linear_models:
            X_tr = fd["X_tr_linear"]
            X_va = fd["X_va_linear"]

        elif name in tree_models:
            X_tr = fd["X_tr_tree"]
            X_va = fd["X_va_tree"]

        elif name in neural_models:
            X_tr = fd["X_tr_mlp"]
            X_va = fd["X_va_mlp"]

        else:
            raise ValueError(f"No feature mapping for {name}")

        model.fit(X_tr, fd["y_tr"])

        p = np.asarray(
            model.predict(X_va)
        ).astype(int).ravel()

        preds[fd["va_idx"]] = p

        fold_score = f1_score(
            fd["y_va"],
            p,
            average="macro",
            zero_division=0
        )
        fold_scores.append(fold_score)

        print(
            f"  Fold {fd['fold']}/{N_SPLITS}: "
            f"Macro-F1={fold_score:.4f}"
        )

    assert (preds >= 0).all(), (
        f"Some rows did not receive OOF predictions for {name}"
    )

    oof_pred[name] = preds
    fold_macro_f1[name] = fold_scores
    run_minutes[name] = (time.time() - t0) / 60

    overall_macro = f1_score(
        y, preds, average="macro", zero_division=0
    )

    print(
        f"\n{name:22s} | "
        f"OOF Macro-F1={overall_macro:.4f} | "
        f"Fold mean±std={np.mean(fold_scores):.4f}"
        f"±{np.std(fold_scores):.4f} | "
        f"Runtime={run_minutes[name]:.2f} min"
    )

# Summary table for all completed models
summary_rows = []

for name, preds in oof_pred.items():
    if len(preds) != len(y) or (np.asarray(preds) < 0).any():
        print(f"Warning: incomplete cached predictions for {name}")
        continue

    summary_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y, preds),
        "Macro-F1": f1_score(
            y, preds, average="macro", zero_division=0
        ),
        "Weighted-F1": f1_score(
            y, preds, average="weighted", zero_division=0
        ),
        "Fold Macro-F1 Mean": np.mean(fold_macro_f1[name]),
        "Fold Macro-F1 Std": np.std(fold_macro_f1[name]),
        "Runtime (min)": run_minutes.get(name, np.nan)
    })

oof_results = pd.DataFrame(summary_rows)

if not oof_results.empty:
    oof_results = oof_results.sort_values(
        "Macro-F1", ascending=False
    ).reset_index(drop=True)

display(oof_results.round(4))



Starting: Logistic Regression


Logistic Regression:   0%|          | 0/5 [00:00<?, ?it/s]

  Fold 1/5: Macro-F1=0.8302
  Fold 2/5: Macro-F1=0.8321
  Fold 3/5: Macro-F1=0.8286
  Fold 4/5: Macro-F1=0.8304
  Fold 5/5: Macro-F1=0.8225

Logistic Regression    | OOF Macro-F1=0.8288 | Fold mean±std=0.8288±0.0033 | Runtime=1.06 min

Starting: Linear SVM


Linear SVM:   0%|          | 0/5 [00:00<?, ?it/s]

  Fold 1/5: Macro-F1=0.8262
  Fold 2/5: Macro-F1=0.8299
  Fold 3/5: Macro-F1=0.8305
  Fold 4/5: Macro-F1=0.8243
  Fold 5/5: Macro-F1=0.8226

Linear SVM             | OOF Macro-F1=0.8267 | Fold mean±std=0.8267±0.0031 | Runtime=0.22 min

Starting: SGD Classifier


...

XGBoost:   0%|          | 0/5 [00:00<?, ?it/s]

  Fold 1/5: Macro-F1=0.8357
  Fold 2/5: Macro-F1=0.8394
  Fold 3/5: Macro-F1=0.8428
  Fold 4/5: Macro-F1=0.8455
  Fold 5/5: Macro-F1=0.8423

XGBoost                | OOF Macro-F1=0.8412 | Fold mean±std=0.8412±0.0033 | Runtime=2.92 min

Starting: MLP Neural Network


MLP Neural Network:   0%|          | 0/5 [00:00<?, ?it/s]

  Fold 1/5: Macro-F1=0.8272
  Fold 2/5: Macro-F1=0.8264
  Fold 3/5: Macro-F1=0.8263
  Fold 4/5: Macro-F1=0.8230
  Fold 5/5: Macro-F1=0.8270

MLP Neural Network     | OOF Macro-F1=0.8260 | Fold mean±std=0.8260±0.0015 | Runtime=0.92 min


,Model,Accuracy,Macro-F1,Weighted-F1,Fold Macro-F1 Mean,Fold Macro-F1 Std,Runtime (min)
0,XGBoost,0.8582,0.8412,0.8610,0.8412,0.0033,2.9200
1,Random Forest,0.8536,0.8356,0.8560,0.8355,0.0046,6.1973
2,Extra Trees,0.8547,0.8350,0.8555,0.8350,0.0021,8.7481
3,Logistic Regression,0.8471,0.8288,0.8487,0.8288,0.0033,1.0632
4,Linear SVM,0.8470,0.8267,0.8467,0.8267,0.0031,0.2225
5,MLP Neural Network,0.8445,0.8260,0.8456,0.8260,0.0015,0.9206
6,SGD Classifier,0.8430,0.8239,0.8444,0.8237,0.0025,0.0935


In [7]:

# ============================================================
# CELL 5 — OOF INTEGRITY CHECK
# ============================================================

print("Dataset rows:", len(y))
print("Saved original row IDs:", len(df_cv["row_id"]))
print("Unique original row IDs:", df_cv["row_id"].nunique())

assert len(df_cv) == len(y)
assert len(oof_fold_id) == len(y)
assert (oof_fold_id >= 1).all()
assert (oof_fold_id <= N_SPLITS).all()

for name, preds in oof_pred.items():
    assert len(preds) == len(y), f"Wrong prediction length: {name}"
    assert (np.asarray(preds) >= 0).all(), f"Missing predictions: {name}"
    assert set(np.unique(preds)).issubset(set(class_ids))

print("\nOOF coverage:")
print(pd.Series(oof_fold_id).value_counts().sort_index())

print("\nModels with complete predictions:")
print(list(oof_pred.keys()))

print("\nFocus-class IDs:")
print({
    c: class_to_id[c]
    for c in FOCUS_CLASSES
    if c in class_to_id
})

print("\nAll integrity checks passed.")


Dataset rows: 42602
Saved original row IDs: 42602
Unique original row IDs: 42602

OOF coverage:
1    8521
2    8521
3    8520
4    8520
5    8520
Name: count, dtype: int64

Models with complete predictions:
['Logistic Regression', 'Linear SVM', 'SGD Classifier', 'Random Forest', 'Extra Trees', 'XGBoost', 'MLP Neural Network']

Focus-class IDs:
{'not_cyberbullying': 3, 'other_cyberbullying': 4}

All integrity checks passed.


In [8]:

# ============================================================
# CELL 7 — PER-CLASS METRICS FOR ALL SEVEN MODELS
# Separate table for each model
# ============================================================

per_class_results = []

for name in model_names:
    if name not in oof_pred:
        continue

    pred = oof_pred[name]

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y,
            pred,
            labels=class_ids,
            zero_division=0
        )
    )

    model_results = []

    for i, class_name in enumerate(class_names):
        row = {
            "Class": class_name,
            "Precision": round(precision[i], 4),
            "Recall": round(recall[i], 4),
            "F1-score": round(f1[i], 4),
            "Support": int(support[i])
        }

        model_results.append(row)

        per_class_results.append({
            "Model": name,
            **row
        })

    # Display a separate table for this model
    model_df = pd.DataFrame(model_results)

    print(f"\n{'=' * 70}")
    print(f"PER-CLASS PERFORMANCE: {name}")
    print("=" * 70)

    display(model_df)

    # Save individual model table
    safe_name = (
        name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    model_df.to_csv(
        f"{safe_name}_per_class_performance.csv",
        index=False
    )

# Combined results for all models
per_class_results_df = pd.DataFrame(per_class_results)

# per_class_results_df.to_csv(
#     "six_class_per_class_performance_all_models.csv",
#     index=False
# )

print("\nAll per-class tables generated successfully.")
print("Individual CSV files saved for each available model.")
print("Combined CSV: six_class_per_class_performance_all_models.csv")



PER-CLASS PERFORMANCE: Logistic Regression


,Class,Precision,Recall,F1-score,Support
0,age,0.9574,0.9715,0.9644,7959
1,ethnicity,0.9752,0.9791,0.9772,7426
2,gender,0.9190,0.8434,0.8796,7478
3,not_cyberbullying,0.5857,0.6272,0.6057,6435
4,other_cyberbullying,0.5914,0.5995,0.5954,5341
5,religion,0.9541,0.9468,0.9504,7963



PER-CLASS PERFORMANCE: Linear SVM


,Class,Precision,Recall,F1-score,Support
0,age,0.9400,0.9791,0.9592,7959
1,ethnicity,0.9661,0.9836,0.9748,7426
2,gender,0.9184,0.8454,0.8804,7478
3,not_cyberbullying,0.5942,0.6020,0.5981,6435
4,other_cyberbullying,0.5966,0.5937,0.5952,5341
5,religion,0.9486,0.9569,0.9527,7963



PER-CLASS PERFORMANCE: SGD Classifier


...


PER-CLASS PERFORMANCE: XGBoost


,Class,Precision,Recall,F1-score,Support
0,age,0.9857,0.9726,0.9791,7959
1,ethnicity,0.9912,0.9830,0.9871,7426
2,gender,0.9228,0.8549,0.8875,7478
3,not_cyberbullying,0.5915,0.6772,0.6315,6435
4,other_cyberbullying,0.6077,0.5975,0.6025,5341
5,religion,0.9670,0.9517,0.9592,7963



PER-CLASS PERFORMANCE: MLP Neural Network


,Class,Precision,Recall,F1-score,Support
0,age,0.9655,0.9727,0.9691,7959
1,ethnicity,0.9754,0.9809,0.9781,7426
2,gender,0.8969,0.8369,0.8659,7478
3,not_cyberbullying,0.5986,0.5728,0.5854,6435
4,other_cyberbullying,0.5732,0.6484,0.6085,5341
5,religion,0.9504,0.9474,0.9489,7963



All per-class tables generated successfully.
Individual CSV files saved for each available model.
Combined CSV: six_class_per_class_performance_all_models.csv


In [9]:

# ============================================================
# CELL 8 — ACCURACY AND CATEGORY-WISE F1-SCORE RANGES
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

# ------------------------------------------------------------
# 1. Accuracy across models
# ------------------------------------------------------------

accuracy_results = []

for name in model_names:
    if name not in oof_pred:
        continue

    pred = oof_pred[name]
    accuracy = accuracy_score(y, pred)

    accuracy_results.append({
        "Model": name,
        "Accuracy (%)": round(accuracy * 100, 2)
    })

accuracy_df = pd.DataFrame(accuracy_results)

print("\n" + "=" * 65)
print("ACCURACY ACROSS MODELS")
print("=" * 65)

display(
    accuracy_df.sort_values(
        "Accuracy (%)", ascending=False
    ).reset_index(drop=True)
)

if not accuracy_df.empty:
    print(
        f"Accuracy range: "
        f"{accuracy_df['Accuracy (%)'].min():.2f}% - "
        f"{accuracy_df['Accuracy (%)'].max():.2f}%"
    )

# ------------------------------------------------------------
# 2. Calculate category-wise F1-score ranges
# ------------------------------------------------------------

category_f1_results = []

for name in model_names:
    if name not in oof_pred:
        continue

    pred = oof_pred[name]

    f1_values = f1_score(
        y,
        pred,
        labels=class_ids,
        average=None,
        zero_division=0
    )

    for i, class_name in enumerate(class_names):
        category_f1_results.append({
            "Model": name,
            "Category": class_name,
            "F1-score (%)": f1_values[i] * 100
        })

category_f1_df = pd.DataFrame(category_f1_results)

# ------------------------------------------------------------
# 3. Summary table: F1-score ranges only
# ------------------------------------------------------------

f1_range_df = (
    category_f1_df
    .groupby("Category")["F1-score (%)"]
    .agg(
        Minimum="min",
        Maximum="max",
        Mean="mean"
    )
    .reset_index()
)

for col in ["Minimum", "Maximum", "Mean"]:
    f1_range_df[col] = f1_range_df[col].round(2)

f1_range_df["F1-score range"] = (
    f1_range_df["Minimum"].map(lambda x: f"{x:.2f}%")
    + " - "
    + f1_range_df["Maximum"].map(lambda x: f"{x:.2f}%")
)

print("\n" + "=" * 65)
print("CATEGORY-WISE F1-SCORE RANGES")
print("=" * 65)

display(
    f1_range_df[
        ["Category", "F1-score range", "Mean"]
    ].rename(columns={
        "Mean": "Mean F1-score (%)"
    })
)

# ------------------------------------------------------------
# 4. Save results
# ------------------------------------------------------------

# accuracy_df.to_csv(
#     "accuracy_across_models.csv",
#     index=False
# )

# f1_range_df.to_csv(
#     "category_wise_f1_ranges.csv",
#     index=False
# )

# print("\nSaved:")
# print("1. accuracy_across_models.csv")
# print("2. category_wise_f1_ranges.csv")



ACCURACY ACROSS MODELS


,Model,Accuracy (%)
0,XGBoost,85.82
1,Extra Trees,85.47
2,Random Forest,85.36
3,Logistic Regression,84.71
4,Linear SVM,84.70
5,MLP Neural Network,84.45
6,SGD Classifier,84.30


Accuracy range: 84.30% - 85.82%

CATEGORY-WISE F1-SCORE RANGES


,Category,F1-score range,Mean F1-score (%)
0,age,95.92% - 97.91%,96.94
1,ethnicity,97.48% - 98.71%,98.04
2,gender,86.59% - 88.75%,87.75
3,not_cyberbullying,58.54% - 63.70%,61.25
4,other_cyberbullying,57.87% - 60.85%,59.23
5,religion,94.89% - 95.92%,95.40


In [10]:

# ============================================================
# CELL 9 — ALL-CLASS ERROR RATE COMPARISON
# ============================================================

import pandas as pd
from sklearn.metrics import accuracy_score

error_rate_results = []

for name in model_names:
    if name not in oof_pred:
        continue

    pred = oof_pred[name]

    accuracy = accuracy_score(y, pred)
    error_rate = 1 - accuracy
    incorrect = (pred != y).sum()
    total = len(y)

    error_rate_results.append({
        "Model": name,
        "Correct Predictions": int(total - incorrect),
        "Incorrect Predictions": int(incorrect),
        "Total Samples": int(total),
        "Accuracy (%)": round(accuracy * 100, 2),
        "Error Rate (%)": round(error_rate * 100, 2)
    })

error_rate_df = pd.DataFrame(error_rate_results)

# Rank models from lowest to highest error rate
error_rate_df = error_rate_df.sort_values(
    "Error Rate (%)",
    ascending=True
).reset_index(drop=True)

print("\n" + "=" * 75)
print("ALL-CLASS ERROR RATE COMPARISON")
print("=" * 75)

display(error_rate_df)

# Identify the lowest and highest error rates
if not error_rate_df.empty:
    best_model = error_rate_df.iloc[0]
    worst_model = error_rate_df.iloc[-1]

    print(
        f"\nLowest error rate: {best_model['Model']} "
        f"({best_model['Error Rate (%)']:.2f}%)"
    )

    print(
        f"Highest error rate: {worst_model['Model']} "
        f"({worst_model['Error Rate (%)']:.2f}%)"
    )

    print(
        f"Error rate range: "
        f"{error_rate_df['Error Rate (%)'].min():.2f}% - "
        f"{error_rate_df['Error Rate (%)'].max():.2f}%"
    )

# Save comparison
# error_rate_df.to_csv(
#     "all_class_error_rate_comparison.csv",
#     index=False
# )

# print("\nSaved: all_class_error_rate_comparison.csv")



ALL-CLASS ERROR RATE COMPARISON


,Model,Correct Predictions,Incorrect Predictions,Total Samples,Accuracy (%),Error Rate (%)
0,XGBoost,36561,6041,42602,85.82,14.18
1,Extra Trees,36411,6191,42602,85.47,14.53
2,Random Forest,36363,6239,42602,85.36,14.64
3,Logistic Regression,36087,6515,42602,84.71,15.29
4,Linear SVM,36084,6518,42602,84.70,15.30
5,MLP Neural Network,35977,6625,42602,84.45,15.55
6,SGD Classifier,35913,6689,42602,84.30,15.70



Lowest error rate: XGBoost (14.18%)
Highest error rate: SGD Classifier (15.70%)
Error rate range: 14.18% - 15.70%


In [11]:

# ============================================================
# CELL 9 — ALL-CLASS ERROR RATE COMPARISON MATRIX
# Rows = Models | Columns = Classes | Values = Error Rate (%)
# ============================================================

import numpy as np
import pandas as pd

error_rate_results = []

for name in model_names:
    if name not in oof_pred:
        continue

    pred = oof_pred[name]

    # Calculate class-wise error rates
    for i, class_name in enumerate(class_names):
        class_mask = np.asarray(y) == class_ids[i]

        total_class_samples = class_mask.sum()

        if total_class_samples == 0:
            error_rate = np.nan
        else:
            incorrect = np.sum(
                np.asarray(pred)[class_mask] != np.asarray(y)[class_mask]
            )

            error_rate = (
                incorrect / total_class_samples
            ) * 100

        error_rate_results.append({
            "Model": name,
            "Class": class_name,
            "Error Rate (%)": round(error_rate, 2)
            if not np.isnan(error_rate) else np.nan
        })

# Convert results into a model-wise matrix
error_rate_df = pd.DataFrame(error_rate_results)

error_rate_matrix = error_rate_df.pivot(
    index="Model",
    columns="Class",
    values="Error Rate (%)"
)

# Preserve model order
error_rate_matrix = error_rate_matrix.reindex(
    [name for name in model_names if name in error_rate_matrix.index]
)

print("\n" + "=" * 75)
print("ALL-CLASS ERROR RATE COMPARISON (%)")
print("=" * 75)

display(error_rate_matrix)

# Save matrix
# error_rate_matrix.to_csv(
#     "all_class_error_rate_comparison.csv"
# )

# print("\nSaved: all_class_error_rate_comparison.csv")



ALL-CLASS ERROR RATE COMPARISON (%)


Class,age,ethnicity,gender,not_cyberbullying,other_cyberbullying,religion
Model,,,,,,
Logistic Regression,2.85,2.09,15.66,37.28,40.05,5.32
Linear SVM,2.09,1.64,15.46,39.80,40.63,4.31
SGD Classifier,3.57,2.60,14.82,38.62,41.57,5.01
Random Forest,2.22,1.87,16.65,30.15,44.06,4.83
Extra Trees,2.04,1.59,15.85,30.12,46.08,4.11
XGBoost,2.74,1.70,14.51,32.28,40.25,4.83
MLP Neural Network,2.73,1.91,16.31,42.72,35.16,5.26


Established other and not cyberbully error rate are the mose

In [12]:

# ============================================================
# CELL 9 — NOT / OTHER ERROR COMPARISON
# ============================================================

import numpy as np
import pandas as pd

y_true = np.asarray(y)

not_cb_id = class_to_id["not_cyberbullying"]
other_cb_id = class_to_id["other_cyberbullying"]

not_other_error_rows = []

for model_name in model_names:
    if model_name not in oof_pred:
        continue

    y_pred = np.asarray(oof_pred[model_name])

    # Total errors across all classes
    total_errors = int(np.sum(y_true != y_pred))
    total_samples = len(y_true)

    # Actual Not-CB samples
    not_cb_mask = (y_true == not_cb_id)
    not_cb_predictions = y_pred[not_cb_mask]

    # Actual Other-CB samples
    other_cb_mask = (y_true == other_cb_id)
    other_cb_predictions = y_pred[other_cb_mask]

    # Actual Not-CB predicted as Other-CB
    not_to_other = int(
        np.sum(not_cb_predictions == other_cb_id)
    )

    # Actual Other-CB predicted as Not-CB
    other_to_not = int(
        np.sum(other_cb_predictions == not_cb_id)
    )

    # Combined errors between Not-CB and Other-CB
    sum_both_errors = not_to_other + other_to_not


    # Percentage of all classification errors that are
    # Not-CB ↔ Other-CB errors
    all_errors_percentage = (
        sum_both_errors / total_errors * 100
        if total_errors > 0 else 0
    )

    not_other_error_rows.append({
        "Model": model_name,
        "Total Error Count": total_errors,
        "Not → Other": not_to_other,
        "Other → Not": other_to_not,
        "Sum of Both Errors": sum_both_errors,
        "All Errors (%)": round(all_errors_percentage, 2)
    })

# Create comparison table
not_other_error_comparison_df = pd.DataFrame(
    not_other_error_rows
)

print("=" * 100)
print("NOT-CYBERBULLYING / OTHER-CYBERBULLYING ERROR COMPARISON")
print("=" * 100)

display(
    not_other_error_comparison_df.sort_values(
        "All Errors (%)",
        ascending=True
    ).reset_index(drop=True)
)

# not_other_error_comparison_df.to_csv(
#     "not_other_error_comparison.csv",
#     index=False
# )

# print("Saved: not_other_error_comparison.csv")


NOT-CYBERBULLYING / OTHER-CYBERBULLYING ERROR COMPARISON


,Model,Total Error Count,Not → Other,Other → Not,Sum of Both Errors,All Errors (%)
0,MLP Neural Network,6625,1874,1368,3242,48.94
1,Linear SVM,6518,1580,1624,3204,49.16
2,SGD Classifier,6689,1622,1733,3355,50.16
3,Logistic Regression,6515,1593,1692,3285,50.42
4,Extra Trees,6191,1230,2048,3278,52.95
5,Random Forest,6239,1333,2022,3355,53.77
6,XGBoost,6041,1505,1850,3355,55.54


In [13]:

# ============================================================
# CELL 9B — OTHER-CB ERROR DESTINATION COMPARISON
# Rows = Models
# Columns = Incorrect prediction destinations
# ============================================================

other_cb_id = class_to_id["other_cyberbullying"]

other_cb_error_rows = []

for model_name in model_names:
    if model_name not in oof_pred:
        continue

    y_pred = np.asarray(oof_pred[model_name])

    # Actual Other-CB samples
    other_cb_mask = (y_true == other_cb_id)

    # Predictions for actual Other-CB samples
    other_cb_predictions = y_pred[other_cb_mask]

    # Keep only incorrect predictions
    incorrect_predictions = other_cb_predictions[
        other_cb_predictions != other_cb_id
    ]

    # Count where each Other-CB error went
    error_counts = {
        class_name: int(
            np.sum(incorrect_predictions == class_id)
        )
        for class_name, class_id in class_to_id.items()
        if class_id != other_cb_id
    }

    total_other_cb_errors = len(incorrect_predictions)

    # Add model and total error count
    row = {
        "Model": model_name,
        "Total Other-CB Errors": total_other_cb_errors
    }

    row.update(error_counts)
    other_cb_error_rows.append(row)

# Create comparison table
other_cb_error_destination_df = pd.DataFrame(
    other_cb_error_rows
)

# Rename destination columns
other_cb_error_destination_df = (
    other_cb_error_destination_df.rename(
        columns={
            class_name: f"Predicted as {class_name}"
            for class_name in class_to_id
            if class_to_id[class_name] != other_cb_id
        }
    )
)

print("=" * 100)
print("OTHER-CYBERBULLYING ERROR DESTINATIONS BY MODEL")
print("=" * 100)

display(other_cb_error_destination_df)

OTHER-CYBERBULLYING ERROR DESTINATIONS BY MODEL


,Model,Total Other-CB Errors,Predicted as age,Predicted as ethnicity,Predicted as gender,Predicted as not_cyberbullying,Predicted as religion
0,Logistic Regression,2139,91,62,235,1692,59
1,Linear SVM,2170,124,87,263,1624,72
2,SGD Classifier,2220,68,52,314,1733,53
3,Random Forest,2353,46,22,243,2022,20
4,Extra Trees,2461,77,43,260,2048,33
5,XGBoost,2150,29,25,230,1850,16
6,MLP Neural Network,1878,73,67,317,1368,53


In [14]:

# ============================================================
# CELL 9 — NOT-CB ERROR DESTINATION COMPARISON
# Rows = Models
# Columns = Incorrect prediction destinations
# ============================================================

import numpy as np
import pandas as pd

not_cb_id = class_to_id["not_cyberbullying"]
y_true = np.asarray(y)

error_rows = []

for model_name in model_names:
    if model_name not in oof_pred:
        continue

    y_pred = np.asarray(oof_pred[model_name])

    # Actual Not-CB samples
    not_cb_mask = (y_true == not_cb_id)

    # Predictions for actual Not-CB samples
    not_cb_predictions = y_pred[not_cb_mask]

    # Keep only incorrect predictions
    incorrect_predictions = not_cb_predictions[
        not_cb_predictions != not_cb_id
    ]

    # Count where each Not-CB error went
    error_counts = {
        class_name: int(
            np.sum(incorrect_predictions == class_id)
        )
        for class_name, class_id in class_to_id.items()
        if class_id != not_cb_id
    }

    total_not_cb_errors = len(incorrect_predictions)

    # Add model and total error count
    row = {
        "Model": model_name,
        "Total Not-CB Errors": total_not_cb_errors
    }

    row.update(error_counts)

    error_rows.append(row)

# Create comparison table
not_cb_error_destination_df = pd.DataFrame(error_rows)

# Rename destination columns for clarity
not_cb_error_destination_df = (
    not_cb_error_destination_df.rename(
        columns={
            class_name: f"Predicted as {class_name}"
            for class_name in class_to_id
            if class_to_id[class_name] != not_cb_id
        }
    )
)

print("=" * 100)
print("NOT-CYBERBULLYING ERROR DESTINATIONS BY MODEL")
print("=" * 100)

display(not_cb_error_destination_df)

NOT-CYBERBULLYING ERROR DESTINATIONS BY MODEL


,Model,Total Not-CB Errors,Predicted as age,Predicted as ethnicity,Predicted as gender,Predicted as other_cyberbullying,Predicted as religion
0,Logistic Regression,2399,225,64,260,1593,257
1,Linear SVM,2561,331,104,249,1580,297
2,SGD Classifier,2485,194,48,325,1622,296
3,Random Forest,1940,119,29,226,1333,233
4,Extra Trees,1938,160,44,248,1230,256
5,XGBoost,2077,69,20,260,1505,223
6,MLP Neural Network,2749,180,58,345,1874,292


#### Not to other and other to not cyberbully is contributing for more then 50% of the error consistently so we can conclude models are not able to distinguish between both of them

Start HNS investigation

In [15]:
# ============================================================
# CELL 15 — CROSS-MODEL CONSISTENCY OF NOT-CB ERRORS
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

NOT_CB = "not_cyberbullying"
not_cb_id = class_to_id[NOT_CB]

# Models with completed OOF predictions
evaluated_models = [
    name for name in model_names if name in oof_pred
]
n_models = len(evaluated_models)

assert n_models > 0, "No models have completed OOF predictions."

# Actual Not-CB samples
y_true = np.asarray(y)
not_cb_mask = (y_true == not_cb_id)
n_not_cb = int(not_cb_mask.sum())

assert n_not_cb > 0, "No actual Not-CB samples found."

# Count models misclassifying each Not-CB sample
errors_per_sample = np.zeros(n_not_cb, dtype=int)

for model_name in evaluated_models:
    predictions = np.asarray(oof_pred[model_name])
    assert len(predictions) == len(y_true), (
        f"Prediction length mismatch: {model_name}"
    )

    errors_per_sample += (
        predictions[not_cb_mask] != not_cb_id
    ).astype(int)

# Distribution of errors across models
counts = (
    pd.Series(errors_per_sample)
    .value_counts()
    .reindex(range(n_models + 1), fill_value=0)
)

table3 = pd.DataFrame({
    "Models Wrong": range(n_models + 1),
    "Samples": counts.to_numpy()
})

table3["Percentage (%)"] = (
    table3["Samples"] / n_not_cb * 100
).round(2)

# Add total row
table3.loc[len(table3)] = [
    "Total",
    int(table3["Samples"].sum()),
    round(table3["Percentage (%)"].sum(), 2)
]

print("TABLE 3 — CROSS-MODEL CONSISTENCY OF NOT-CB ERRORS")
print(f"Evaluated models: {n_models}")
print(f"Actual Not-CB samples: {n_not_cb:,}")

display(table3)

# Samples misclassified by every model
persistent_count = int((errors_per_sample == n_models).sum())
persistent_pct = persistent_count / n_not_cb * 100

print(
    f"Misclassified by all {n_models} models: "
    f"{persistent_count:,} ({persistent_pct:.2f}%)"
)

TABLE 3 — CROSS-MODEL CONSISTENCY OF NOT-CB ERRORS
Evaluated models: 7
Actual Not-CB samples: 6,435


,Models Wrong,Samples,Percentage (%)
0,0,2373,36.88
1,1,812,12.62
2,2,489,7.60
3,3,537,8.34
4,4,510,7.93
5,5,418,6.50
6,6,454,7.06
7,7,842,13.08
8,Total,6435,100.01


Misclassified by all 7 models: 842 (13.08%)


In [16]:
# ============================================================
# CELL 16 — CROSS-MODEL CONSISTENCY OF OTHER-CB ERRORS
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

OTHER_CB = "other_cyberbullying"
other_cb_id = class_to_id[OTHER_CB]

# Models with completed OOF predictions
evaluated_models = [
    name for name in model_names if name in oof_pred
]
n_models = len(evaluated_models)

assert n_models > 0, "No models have completed OOF predictions."

# Actual Other-CB samples
y_true = np.asarray(y)
other_cb_mask = (y_true == other_cb_id)
n_other_cb = int(other_cb_mask.sum())

assert n_other_cb > 0, "No actual Other-CB samples found."

# Count models misclassifying each Other-CB sample
errors_per_sample = np.zeros(n_other_cb, dtype=int)

for model_name in evaluated_models:
    predictions = np.asarray(oof_pred[model_name])

    assert len(predictions) == len(y_true), (
        f"Prediction length mismatch: {model_name}"
    )

    errors_per_sample += (
        predictions[other_cb_mask] != other_cb_id
    ).astype(int)

# Distribution of errors across models
counts = (
    pd.Series(errors_per_sample)
    .value_counts()
    .reindex(range(n_models + 1), fill_value=0)
)

table4 = pd.DataFrame({
    "Models Wrong": range(n_models + 1),
    "Samples": counts.to_numpy()
})

table4["Percentage (%)"] = (
    table4["Samples"] / n_other_cb * 100
).round(2)

# Add total row
table4.loc[len(table4)] = [
    "Total",
    int(table4["Samples"].sum()),
    round(table4["Percentage (%)"].sum(), 2)
]

print("TABLE 4 — CROSS-MODEL CONSISTENCY OF OTHER-CB ERRORS")
print(f"Evaluated models: {n_models}")
print(f"Actual Other-CB samples: {n_other_cb:,}")

display(table4)

# Samples misclassified by every model
persistent_count = int((errors_per_sample == n_models).sum())
persistent_pct = persistent_count / n_other_cb * 100

print(
    f"Misclassified by all {n_models} models: "
    f"{persistent_count:,} ({persistent_pct:.2f}%)"
)

TABLE 4 — CROSS-MODEL CONSISTENCY OF OTHER-CB ERRORS
Evaluated models: 7
Actual Other-CB samples: 5,341


,Models Wrong,Samples,Percentage (%)
0,0,1584,29.66
1,1,579,10.84
2,2,527,9.87
3,3,518,9.70
4,4,491,9.19
5,5,396,7.41
6,6,482,9.02
7,7,764,14.30
8,Total,5341,99.99


Misclassified by all 7 models: 764 (14.30%)


In [17]:

# ============================================================
# CELL 3 — CREATE FOUR PERSISTENT GROUPS
# Uses CELL 1 variables and completed OOF predictions
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

# ---------------- CONFIG ----------------
NOT_CB = "not_cyberbullying"
OTHER_CB = "other_cyberbullying"

MIN_WRONG_MODELS = 4
MAX_WRONG_MODELS_CORRECT = 1

# ---------------- VALIDATION ----------------
evaluated_models = [
    name for name in model_names if name in oof_pred
]
n_models = len(evaluated_models)

assert n_models >= MIN_WRONG_MODELS, (
    f"At least {MIN_WRONG_MODELS} evaluated models required; "
    f"found {n_models}."
)

assert NOT_CB in class_to_id, f"Missing class: {NOT_CB}"
assert OTHER_CB in class_to_id, f"Missing class: {OTHER_CB}"

y_true = np.asarray(y).ravel()

assert len(y_true) == len(df_cv), (
    "The lengths of y and df_cv do not match."
)

not_cb_id = class_to_id[NOT_CB]
other_cb_id = class_to_id[OTHER_CB]

# ---------------- BUILD OOF PREDICTION MATRIX ----------------
# Rows = samples; columns = evaluated models

pred_matrix = np.column_stack([
    np.asarray(oof_pred[name]).ravel()
    for name in evaluated_models
])

assert pred_matrix.shape == (len(y_true), n_models), (
    f"Unexpected prediction matrix shape: {pred_matrix.shape}"
)

# Total models that incorrectly predicted each sample
wrong_count = (pred_matrix != y_true[:, None]).sum(axis=1)

# Specific cross-classification counts
not_to_other_count = (
    pred_matrix == other_cb_id
).sum(axis=1)

other_to_not_count = (
    pred_matrix == not_cb_id
).sum(axis=1)

# ---------------- KEEP THE TWO ACTUAL CLASSES ----------------
target_mask = np.isin(y_true, [not_cb_id, other_cb_id])

# ---------------- ASSIGN GROUP LABELS ----------------
group_labels = np.full(
    len(y_true),
    "Not in persistent groups",
    dtype=object
)

# 1. Actual Not-CB predicted as Other-CB by >= 4 models
mask_incorrect_not = (
    (y_true == not_cb_id)
    & (not_to_other_count >= MIN_WRONG_MODELS)
)

group_labels[mask_incorrect_not] = (
    "Persistently incorrect Not-CB"
)

# 2. Actual Other-CB predicted as Not-CB by >= 4 models
mask_incorrect_other = (
    (y_true == other_cb_id)
    & (other_to_not_count >= MIN_WRONG_MODELS)
)

group_labels[mask_incorrect_other] = (
    "Persistently incorrect Other-CB"
)

# 3. Actual Not-CB with at most 1 incorrect model
mask_correct_not = (
    (y_true == not_cb_id)
    & (wrong_count <= MAX_WRONG_MODELS_CORRECT)
)

group_labels[mask_correct_not] = (
    "Persistently correct Not-CB"
)

# 4. Actual Other-CB with at most 1 incorrect model
mask_correct_other = (
    (y_true == other_cb_id)
    & (wrong_count <= MAX_WRONG_MODELS_CORRECT)
)

group_labels[mask_correct_other] = (
    "Persistently correct Other-CB"
)

# ---------------- CREATE NEW DATAFRAME ----------------
# Preserve original df_cv columns and original row_id.
persistent_df = df_cv.loc[target_mask].copy()

# Attach prediction statistics using the original positional order.
persistent_df["true_label"] = [
    NOT_CB if label == not_cb_id else OTHER_CB
    for label in y_true[target_mask]
]

persistent_df["wrong_model_count"] = wrong_count[target_mask]
persistent_df["not_to_other_count"] = not_to_other_count[target_mask]
persistent_df["other_to_not_count"] = other_to_not_count[target_mask]

persistent_df["persistent_group"] = group_labels[target_mask]

# Add each model's prediction as a readable class label.
for j, model_name in enumerate(evaluated_models):
    persistent_df[f"pred_{model_name}"] = [
        class_names[int(pred)]
        for pred in pred_matrix[target_mask, j]
    ]

# ---------------- SUMMARY ----------------
group_order = [
    "Persistently incorrect Not-CB",
    "Persistently incorrect Other-CB",
    "Persistently correct Not-CB",
    "Persistently correct Other-CB",
    "Not in persistent groups"
]

group_summary = (
    persistent_df["persistent_group"]
    .value_counts()
    .reindex(group_order, fill_value=0)
    .rename_axis("persistent_group")
    .reset_index(name="sample_count")
)

group_summary["percentage_of_two_classes (%)"] = (
    group_summary["sample_count"]
    / len(persistent_df) * 100
).round(2)

print("Evaluated models:", n_models)
print("Not-CB + Other-CB samples:", len(persistent_df))
print("\nGROUP SUMMARY")
display(group_summary)

# ---------------- DATAFRAME FOR NEXT ANALYSIS ----------------
persistent_four_df = persistent_df[
    persistent_df["persistent_group"].isin(group_order[:4])
].copy()

print("\nFour-group dataframe shape:", persistent_four_df.shape)
print("\nSamples per persistent group:")
display(
    persistent_four_df["persistent_group"]
    .value_counts()
    .reindex(group_order[:4], fill_value=0)
    .rename_axis("persistent_group")
    .reset_index(name="sample_count")
)

display(
    persistent_four_df[
        [
            "row_id",
            RAW_TEXT_COL,
            TEXT_COL,
            TARGET_COL,
            "true_label",
            "wrong_model_count",
            "persistent_group"
        ]
    ].head(10)
)


Evaluated models: 7
Not-CB + Other-CB samples: 11776

GROUP SUMMARY


,persistent_group,sample_count,percentage_of_two_classes (%)
0,Persistently incorrect Not-CB,1432,12.16
1,Persistently incorrect Other-CB,1671,14.19
2,Persistently correct Not-CB,3185,27.05
3,Persistently correct Other-CB,2163,18.37
4,Not in persistent groups,3325,28.24



Four-group dataframe shape: (8451, 27)

Samples per persistent group:


,persistent_group,sample_count
0,Persistently incorrect Not-CB,1432
1,Persistently incorrect Other-CB,1671
2,Persistently correct Not-CB,3185
3,Persistently correct Other-CB,2163


,row_id,tweet_text,lemmatized_tweet,cyberbullying_type,true_label,wrong_model_count,persistent_group
0,0,"in other words #katandandre, your food was cra...",word katandandre food crapilicious mkr,not_cyberbullying,not_cyberbullying,0,Persistently correct Not-CB
1,1,why is #aussietv so white? #mkr #theblock #ima...,aussietv white mkr theblock imacelebrityau tod...,not_cyberbullying,not_cyberbullying,0,Persistently correct Not-CB
2,2,@xochitlsuckkks a classy whore? or more red ve...,classy whore red velvet cupcake,not_cyberbullying,not_cyberbullying,1,Persistently correct Not-CB
3,3,"@jason_gio meh. :p thanks for the heads up, b...",meh p thanks head concerned another angry dude...,not_cyberbullying,not_cyberbullying,6,Persistently incorrect Not-CB
5,5,"@raja5aab @quickieleaks yes, the test of god i...",yes test god good bad indifferent weird whatev...,not_cyberbullying,not_cyberbullying,6,Persistently incorrect Not-CB
6,6,@stockputout everything but mostly my priest,everything mostly priest,not_cyberbullying,not_cyberbullying,7,Persistently incorrect Not-CB
9,9,rt @kurdsnews: turkish state has killed 241 ch...,rt turkish state killed child last year news g...,not_cyberbullying,not_cyberbullying,0,Persistently correct Not-CB
10,10,love that the best response to the hotcakes th...,love best response hotcake managed film noncom...,not_cyberbullying,not_cyberbullying,0,Persistently correct Not-CB
11,11,@0xabad1dea @kelseytheodore2 twitter is basica...,twitter basically angry letter generation,not_cyberbullying,not_cyberbullying,6,Persistently incorrect Not-CB
12,12,"best pick up line? hi, you're cute... ?: i lov...",best pick line hi youre cute love people call ...,not_cyberbullying,not_cyberbullying,6,Persistently incorrect Not-CB


In [18]:

# ============================================================
# CELL 4 — CLASS-WISE TF-IDF FOR FOUR PERSISTENT GROUPS
# Top-K features + pairwise overlap
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer

# ---------------- CONFIGURATION ----------------
TOP_K = 50

TEXT_COL = "lemmatized_tweet"

TFIDF_ANALYZER = "word"
TFIDF_NGRAM_RANGE = (1, 2)
TFIDF_MAX_FEATURES = 10000
TFIDF_SUBLINEAR_TF = False

GROUPS = [
    "Persistently incorrect Not-CB",
    "Persistently incorrect Other-CB",
    "Persistently correct Not-CB",
    "Persistently correct Other-CB",
]

# ---------------- VALIDATE DATA ----------------
assert "persistent_four_df" in globals(), (
    "Run Cell 3 first to create persistent_four_df."
)

assert TEXT_COL in persistent_four_df.columns, (
    f"Missing text column: {TEXT_COL}"
)

assert "persistent_group" in persistent_four_df.columns

# Work on a copy; preserve original dataframe
tfidf_df = persistent_four_df.copy()

tfidf_df[TEXT_COL] = (
    tfidf_df[TEXT_COL].fillna("").astype(str)
)

tfidf_df = tfidf_df[
    tfidf_df["persistent_group"].isin(GROUPS)
].copy()

# ---------------- CLASS-WISE TF-IDF ----------------
# Fit a separate TF-IDF vectorizer for each group.
# Rank features by mean TF-IDF across that group's documents.

top_k_features = {}
tfidf_matrices = {}
tfidf_vectorizers = {}
group_document_counts = {}

for group_name in GROUPS:

    group_df = tfidf_df[
        tfidf_df["persistent_group"] == group_name
    ]

    texts = group_df[TEXT_COL].tolist()
    group_document_counts[group_name] = len(texts)

    print(f"\n{'=' * 70}")
    print(f"GROUP: {group_name}")
    print(f"Documents: {len(texts)}")

    if len(texts) == 0:
        print("No samples in this group; skipping.")
        top_k_features[group_name] = pd.DataFrame(
            columns=["Feature", "Mean_TFIDF", "Rank"]
        )
        continue

    # Empty documents cannot contribute terms.
    nonempty_texts = [text for text in texts if text.strip()]

    if not nonempty_texts:
        print("No non-empty text documents; skipping.")
        top_k_features[group_name] = pd.DataFrame(
            columns=["Feature", "Mean_TFIDF", "Rank"]
        )
        continue

    vectorizer = TfidfVectorizer(
        analyzer=TFIDF_ANALYZER,
        ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES,
        sublinear_tf=TFIDF_SUBLINEAR_TF,
        dtype=np.float32
    )

    try:
        X_group_tfidf = vectorizer.fit_transform(nonempty_texts)
    except ValueError as exc:
        print(f"TF-IDF could not be fitted: {exc}")
        top_k_features[group_name] = pd.DataFrame(
            columns=["Feature", "Mean_TFIDF", "Rank"]
        )
        continue

    feature_names = vectorizer.get_feature_names_out()

    # Mean TF-IDF score per feature across documents in this group
    mean_scores = np.asarray(
        X_group_tfidf.mean(axis=0)
    ).ravel()

    ranked_idx = np.argsort(-mean_scores, kind="stable")[:TOP_K]

    top_df = pd.DataFrame({
        "Feature": feature_names[ranked_idx],
        "Mean_TFIDF": mean_scores[ranked_idx],
        "Rank": np.arange(1, len(ranked_idx) + 1)
    })

    top_k_features[group_name] = top_df
    tfidf_matrices[group_name] = X_group_tfidf
    tfidf_vectorizers[group_name] = vectorizer

    print(f"Vocabulary size: {len(feature_names)}")
    print(f"Top {TOP_K} TF-IDF features:")
    display(top_df)

# ---------------- PAIRWISE TOP-K FEATURE OVERLAP ----------------
def compare_top_k(group_a, group_b):
    df_a = top_k_features[group_a]
    df_b = top_k_features[group_b]

    set_a = set(df_a["Feature"])
    set_b = set(df_b["Feature"])

    common = set_a & set_b
    union = set_a | set_b

    jaccard = len(common) / len(union) if union else 0.0

    # Preserve rank and score from each group's TF-IDF table
    common_df = (
        df_a[df_a["Feature"].isin(common)]
        .merge(
            df_b,
            on="Feature",
            how="inner",
            suffixes=("_A", "_B")
        )
        .sort_values(["Rank_A", "Rank_B"])
    )

    overlap_details = "; ".join(common_df["Feature"].tolist())

    return {
        "Group A": group_a,
        "Group B": group_b,
        "Top-K A features": len(set_a),
        "Top-K B features": len(set_b),
        "Shared features": len(common),
        "Union features": len(union),
        "Jaccard overlap (%)": round(jaccard * 100, 2),
        "Shared terms": overlap_details
    }, common_df

# Compare the key pairs requested:
# 1. Incorrect Not-CB vs Correct Not-CB
# 2. Incorrect Other-CB vs Correct Other-CB
# 3. Both incorrect groups
# 4. Both correct groups

PAIRWISE_COMPARISONS = [
    (GROUPS[0], GROUPS[2]),
    (GROUPS[1], GROUPS[3]),
    (GROUPS[0], GROUPS[1]),
    (GROUPS[2], GROUPS[3]),
]

overlap_rows = []
overlap_feature_tables = {}

for group_a, group_b in PAIRWISE_COMPARISONS:
    result, shared_df = compare_top_k(group_a, group_b)

    overlap_rows.append(result)
    overlap_feature_tables[(group_a, group_b)] = shared_df

overlap_summary = pd.DataFrame(overlap_rows)

print("\n" + "=" * 70)
print("TOP-K TF-IDF FEATURE OVERLAP SUMMARY")
display(
    overlap_summary[
        [
            "Group A",
            "Group B",
            "Top-K A features",
            "Top-K B features",
            "Shared features",
            "Union features",
            "Jaccard overlap (%)"
        ]
    ]
)

# ---------------- DISPLAY SHARED FEATURES WITH RANKS ----------------
for (group_a, group_b), shared_df in overlap_feature_tables.items():

    print("\n" + "-" * 70)
    print("SHARED FEATURES")
    print(f"A: {group_a}")
    print(f"B: {group_b}")

    if shared_df.empty:
        print("No shared top-K features.")
    else:
        display(shared_df)

# ---------------- OPTIONAL: SAVE RESULTS ----------------
# Uncomment to export:
#
# overlap_summary.to_csv(
#     "persistent_tfidf_overlap_summary.csv", index=False
# )
#
# for i, group_name in enumerate(GROUPS, start=1):
#     top_k_features[group_name].to_csv(
#         f"persistent_group_{i}_top_tfidf.csv", index=False
#     )

print("\nTF-IDF feature and overlap analysis completed.")



GROUP: Persistently incorrect Not-CB
Documents: 1432
Vocabulary size: 10000
Top 50 TF-IDF features:


,Feature,Mean_TFIDF,Rank
0,bully,0.016914,1
1,im,0.016858,2
2,rt,0.013364,3
3,dont,0.011725,4
4,people,0.010289,5
5,know,0.009212,6
6,like,0.009107,7
7,get,0.008394,8
8,think,0.008325,9
9,thing,0.007992,10



GROUP: Persistently incorrect Other-CB
Documents: 1671
Vocabulary size: 10000
Top 50 TF-IDF features:


,Feature,Mean_TFIDF,Rank
0,rt,0.013271,1
1,mkr,0.013029,2
2,bullying,0.012446,3
3,im,0.011116,4
4,dont,0.008991,5
5,know,0.008190,6
6,like,0.007984,7
7,get,0.007927,8
8,think,0.007824,9
9,one,0.006356,10



GROUP: Persistently correct Not-CB
Documents: 3185
Vocabulary size: 10000
Top 50 TF-IDF features:


...


----------------------------------------------------------------------
SHARED FEATURES
A: Persistently incorrect Not-CB
B: Persistently incorrect Other-CB


,Feature,Mean_TFIDF_A,Rank_A,Mean_TFIDF_B,Rank_B
0,bully,0.016914,1,0.004927,23
1,im,0.016858,2,0.011116,4
2,rt,0.013364,3,0.013271,1
3,dont,0.011725,4,0.008991,5
4,people,0.010289,5,0.005544,18
5,know,0.009212,6,0.008190,6
6,like,0.009107,7,0.007984,7
7,get,0.008394,8,0.007927,8
8,think,0.008325,9,0.007824,9
9,thing,0.007992,10,0.004246,34



----------------------------------------------------------------------
SHARED FEATURES
A: Persistently correct Not-CB
B: Persistently correct Other-CB


,Feature,Mean_TFIDF_A,Rank_A,Mean_TFIDF_B,Rank_B
0,rt,0.016905,2,0.021909,2
1,im,0.008906,5,0.018840,4
2,amp,0.008406,8,0.005365,36
3,one,0.008399,9,0.006018,27
4,get,0.008346,10,0.010017,11
5,like,0.007974,12,0.010314,9
6,go,0.007826,13,0.004868,42
7,time,0.007461,15,0.006139,25
8,lol,0.007234,16,0.006609,20
9,dont,0.007214,17,0.011463,8



TF-IDF feature and overlap analysis completed.


### one shared CountVectorizer vocabulary and pairwise log-odds z-scores with an informative Dirichlet prior, instead of ranking each group independently by mean TF-IDF.

![image.png](attachment:9339bb13-3835-428f-8755-0f4ec6b613d2.png)

In [19]:



# ============================================================
# CELL 4 — PAIRWISE DISTINCTIVE FEATURES FOR FOUR PERSISTENT GROUPS
# Shared CountVectorizer + informative Dirichlet prior
# Monroe et al. (2008) log-odds z-score
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer

# ---------------- CONFIGURATION ----------------
TOP_K = 50

TEXT_COL = "lemmatized_tweet"
GROUP_COL = "persistent_group"

NGRAM_RANGE = (1, 2)
MAX_FEATURES = 10000
MIN_DF = 1

# Total concentration of the informative Dirichlet prior.
# A conventional starting value; test sensitivity before
# treating rankings as robust research findings.
PRIOR_STRENGTH = 1000.0

# Remove noise tokens that survived earlier preprocessing.
CUSTOM_STOP_WORDS = {
    "rt", "amp",
    "im", "dont", "ive"
}

GROUPS = [
    "Persistently incorrect Not-CB",
    "Persistently incorrect Other-CB",
    "Persistently correct Not-CB",
    "Persistently correct Other-CB",
]

PAIRWISE_COMPARISONS = [
    (GROUPS[0], GROUPS[2]),  # Incorrect vs correct Not-CB
    (GROUPS[1], GROUPS[3]),  # Incorrect vs correct Other-CB
    (GROUPS[0], GROUPS[1]),  # Incorrect Not-CB vs incorrect Other-CB
    (GROUPS[2], GROUPS[3]),  # Correct Not-CB vs correct Other-CB
]

# ---------------- VALIDATE DATA ----------------
assert "persistent_four_df" in globals(), (
    "Run Cell 3 first to create persistent_four_df."
)
assert TEXT_COL in persistent_four_df.columns, (
    f"Missing text column: {TEXT_COL}"
)
assert GROUP_COL in persistent_four_df.columns, (
    f"Missing group column: {GROUP_COL}"
)

# Work on a copy; preserve the original dataframe.
count_df = persistent_four_df[
    persistent_four_df[GROUP_COL].isin(GROUPS)
].copy()

count_df[TEXT_COL] = (
    count_df[TEXT_COL].fillna("").astype(str)
)

count_df = count_df.reset_index(drop=True)

print("Documents per group:")
display(
    count_df[GROUP_COL]
    .value_counts()
    .reindex(GROUPS, fill_value=0)
    .rename("Documents")
    .rename_axis("Group")
    .reset_index()
)

# ---------------- ONE SHARED COUNT VECTORIZER ----------------
# Every group uses the same vocabulary.
# Counts, not TF-IDF weights, are required for log-odds.

vectorizer = CountVectorizer(
    analyzer="word",
    ngram_range=NGRAM_RANGE,
    max_features=MAX_FEATURES,
    min_df=MIN_DF,
    stop_words=sorted(CUSTOM_STOP_WORDS),
    dtype=np.int64
)

try:
    X_counts = vectorizer.fit_transform(count_df[TEXT_COL])
except ValueError as exc:
    raise ValueError(
        "CountVectorizer could not fit. Check the text and "
        "stop-word configuration."
    ) from exc

feature_names = vectorizer.get_feature_names_out()
vocab_size = len(feature_names)

print(f"\nShared vocabulary size: {vocab_size}")
print(f"Count matrix shape: {X_counts.shape}")
print(f"Custom stop words: {sorted(CUSTOM_STOP_WORDS)}")

# Map each group to row positions in the shared matrix.
group_indices = {
    group: np.flatnonzero(
        count_df[GROUP_COL].to_numpy() == group
    )
    for group in GROUPS
}

# Aggregate token counts across documents in each group.
group_token_counts = {}
group_total_tokens = {}
group_document_counts = {}

for group in GROUPS:
    rows = group_indices[group]

    counts = np.asarray(
        X_counts[rows].sum(axis=0)
    ).ravel().astype(np.float64)

    group_token_counts[group] = counts
    group_total_tokens[group] = counts.sum()
    group_document_counts[group] = len(rows)

# ---------------- INFORMATIVE DIRICHLET PRIOR ----------------
# Estimate the background word distribution from all four groups.
# alpha_w = prior strength * background frequency of word w.
#
# This provides a shared informative prior for every comparison.

pooled_counts = np.sum(
    np.vstack([group_token_counts[g] for g in GROUPS]),
    axis=0
)

pooled_total = pooled_counts.sum()

if pooled_total <= 0:
    raise ValueError("No tokens remain after preprocessing.")

background_probs = pooled_counts / pooled_total
alpha_w = PRIOR_STRENGTH * background_probs
alpha_0 = alpha_w.sum()

print(f"Total pooled tokens: {int(pooled_total):,}")
print(f"Prior concentration: {alpha_0:.2f}")

# ---------------- PAIRWISE LOG-ODDS Z-SCORE ----------------
def compare_log_odds(group_a, group_b):
    """
    Positive z: term is more associated with group A.
    Negative z: term is more associated with group B.

    Uses the informative Dirichlet prior and the approximate
    variance described by Monroe et al. (2008).
    """

    a = group_token_counts[group_a]
    b = group_token_counts[group_b]

    n_a = group_total_tokens[group_a]
    n_b = group_total_tokens[group_b]

    if n_a <= 0 or n_b <= 0:
        raise ValueError(
            f"One comparison group has no tokens: "
            f"{group_a} / {group_b}"
        )

    # Log odds for each word versus all other words.
    log_odds_a = (
        np.log(a + alpha_w)
        - np.log(n_a + alpha_0 - a - alpha_w)
    )

    log_odds_b = (
        np.log(b + alpha_w)
        - np.log(n_b + alpha_0 - b - alpha_w)
    )

    delta = log_odds_a - log_odds_b

    # Approximate variance under the informative prior.
    variance = (
        1.0 / (a + alpha_w)
        + 1.0 / (b + alpha_w)
    )

    z_scores = delta / np.sqrt(variance)

    result_df = pd.DataFrame({
        "Feature": feature_names,
        "Count_A": a.astype(np.int64),
        "Count_B": b.astype(np.int64),
        "Log_Odds_Difference": delta,
        "Z_Score": z_scores
    })

    result_df["Abs_Z_Score"] = (
        result_df["Z_Score"].abs()
    )

    result_df["More_Associated_With"] = np.where(
        result_df["Z_Score"] > 0,
        group_a,
        group_b
    )

    # Return strongest terms on each side of the comparison.
    a_top = (
        result_df[result_df["Z_Score"] > 0]
        .sort_values("Z_Score", ascending=False)
        .head(TOP_K)
        .copy()
    )

    b_top = (
        result_df[result_df["Z_Score"] < 0]
        .sort_values("Z_Score", ascending=True)
        .head(TOP_K)
        .copy()
    )

    a_top["Rank"] = np.arange(1, len(a_top) + 1)
    b_top["Rank"] = np.arange(1, len(b_top) + 1)

    return result_df, a_top, b_top


# ---------------- RUN ALL FOUR COMPARISONS ----------------
pairwise_logodds_tables = {}
pairwise_feature_tables = {}
logodds_summary_rows = []

for group_a, group_b in PAIRWISE_COMPARISONS:

    print("\n" + "=" * 80)
    print(f"GROUP A: {group_a}")
    print(f"GROUP B: {group_b}")

    all_features, top_a, top_b = compare_log_odds(
        group_a, group_b
    )

    pair_key = (group_a, group_b)

    pairwise_logodds_tables[pair_key] = all_features
    pairwise_feature_tables[pair_key] = {
        group_a: top_a,
        group_b: top_b
    }

    print(f"\nTop distinctive features for GROUP A: {group_a}")
    display(
        top_a[[
            "Rank", "Feature", "Count_A", "Count_B",
            "Log_Odds_Difference", "Z_Score"
        ]].round(4)
    )

    print(f"\nTop distinctive features for GROUP B: {group_b}")
    display(
        top_b[[
            "Rank", "Feature", "Count_A", "Count_B",
            "Log_Odds_Difference", "Z_Score"
        ]].round(4)
    )

    # Summary of the strongest directional contrasts.
    logodds_summary_rows.append({
        "Group_A": group_a,
        "Group_B": group_b,
        "Documents_A": group_document_counts[group_a],
        "Documents_B": group_document_counts[group_b],
        "Tokens_A": int(group_total_tokens[group_a]),
        "Tokens_B": int(group_total_tokens[group_b]),
        "Top_A_Features": len(top_a),
        "Top_B_Features": len(top_b),
        "Max_Abs_Z": all_features["Abs_Z_Score"].max(),
        "Features_Abs_Z_ge_2": int(
            (all_features["Abs_Z_Score"] >= 2).sum()
        ),
        "Features_Abs_Z_ge_3": int(
            (all_features["Abs_Z_Score"] >= 3).sum()
        )
    })

# ---------------- COMPARISON SUMMARY ----------------
logodds_summary = pd.DataFrame(logodds_summary_rows)

print("\n" + "=" * 80)
print("PAIRWISE LOG-ODDS SUMMARY")
display(logodds_summary.round(3))

# ---------------- TOP-FEATURE OVERLAP ----------------
# For each comparison, compare the top distinctive terms on
# the A side with those on the B side.
# This overlap is descriptive, not a significance test.

overlap_rows = []
overlap_feature_tables = {}

for group_a, group_b in PAIRWISE_COMPARISONS:

    top_a = pairwise_feature_tables[(group_a, group_b)][group_a]
    top_b = pairwise_feature_tables[(group_a, group_b)][group_b]

    set_a = set(top_a["Feature"])
    set_b = set(top_b["Feature"])

    common = set_a & set_b
    union = set_a | set_b

    jaccard = len(common) / len(union) if union else 0.0

    overlap_rows.append({
        "Group_A": group_a,
        "Group_B": group_b,
        "Top_K": TOP_K,
        "Shared_Top_Features": len(common),
        "Union_Top_Features": len(union),
        "Jaccard_Overlap_Pct": 100 * jaccard
    })

    overlap_feature_tables[(group_a, group_b)] = sorted(common)

overlap_summary = pd.DataFrame(overlap_rows)

print("\nTOP-K DISTINCTIVE FEATURE OVERLAP SUMMARY")
display(overlap_summary.round(2))

# ---------------- OPTIONAL: EXPORT RESULTS ----------------
# logodds_summary.to_csv(
#     "persistent_logodds_summary.csv", index=False
# )
# overlap_summary.to_csv(
#     "persistent_logodds_overlap_summary.csv", index=False
# )
#
# for i, (pair, tables) in enumerate(
#     pairwise_feature_tables.items(), start=1
# ):
#     for group_name, table in tables.items():
#         safe_name = (
#             group_name.lower()
#             .replace(" ", "_")
#             .replace("-", "_")
#         )
#         table.to_csv(
#             f"logodds_pair_{i}_{safe_name}.csv",
#             index=False
#         )

print("\nShared-vocabulary log-odds analysis completed.")


Documents per group:


,Group,Documents
0,Persistently incorrect Not-CB,1432
1,Persistently incorrect Other-CB,1671
2,Persistently correct Not-CB,3185
3,Persistently correct Other-CB,2163



Shared vocabulary size: 10000
Count matrix shape: (8451, 10000)
Custom stop words: ['amp', 'dont', 'im', 'ive', 'rt']
Total pooled tokens: 67,389
Prior concentration: 1000.00

GROUP A: Persistently incorrect Not-CB
GROUP B: Persistently correct Not-CB

Top distinctive features for GROUP A: Persistently incorrect Not-CB


,Rank,Feature,Count_A,Count_B,Log_Odds_Difference,Z_Score
1002,1,bully,152,39,2.0282,12.1330
6755,2,people,102,87,0.9336,6.6111
8827,3,thing,64,39,1.2472,6.3586
3992,4,fucking,41,11,1.8229,6.3434
9194,5,twitter,51,23,1.5201,6.3347
983,6,bullied,36,4,2.5753,6.0272
5313,7,know,76,69,0.8739,5.4190
4068,8,gamergate,32,2,3.1123,5.3413
457,9,as,27,4,2.3448,5.2627
4560,10,harassment,28,2,3.0387,5.0398



Top distinctive features for GROUP B: Persistently correct Not-CB


...


PAIRWISE LOG-ODDS SUMMARY


,Group_A,Group_B,Documents_A,Documents_B,Tokens_A,Tokens_B,Top_A_Features,Top_B_Features,Max_Abs_Z,Features_Abs_Z_ge_2,Features_Abs_Z_ge_3
0,Persistently incorrect Not-CB,Persistently correct Not-CB,1432,3185,11337,25892,50,50,15.908,240,77
1,Persistently incorrect Other-CB,Persistently correct Other-CB,1671,2163,11959,18201,50,50,10.148,123,34
2,Persistently incorrect Not-CB,Persistently incorrect Other-CB,1432,1671,11337,11959,50,50,8.120,43,11
3,Persistently correct Not-CB,Persistently correct Other-CB,3185,2163,25892,18201,50,50,17.719,288,109



TOP-K DISTINCTIVE FEATURE OVERLAP SUMMARY


,Group_A,Group_B,Top_K,Shared_Top_Features,Union_Top_Features,Jaccard_Overlap_Pct
0,Persistently incorrect Not-CB,Persistently correct Not-CB,50,0,100,0.0
1,Persistently incorrect Other-CB,Persistently correct Other-CB,50,0,100,0.0
2,Persistently incorrect Not-CB,Persistently incorrect Other-CB,50,0,100,0.0
3,Persistently correct Not-CB,Persistently correct Other-CB,50,0,100,0.0



Shared-vocabulary log-odds analysis completed.


Option C: Fit a new TF-IDF with a controlled vocabulary

My preferred starting point for your analysis.

Fit TF-IDF on all four groups together, but control the vocabulary using document-frequency thresholds and an appropriate feature limit.

Reduces the influence of extremely rare terms.

Can reduce noise from spelling variations, usernames and other artifacts.

Retains a common feature space for every group.

Requires sensitivity checks to make sure feature selection is not changing the conclusions.

In [20]:
# import numpy as np, pandas as pd, re
# from IPython.display import display
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.preprocessing import normalize

# # ---------------- CONFIG ----------------
# TEXT_COL = "lemmatized_tweet"
# GROUP_COL = "persistent_group"
# N_BOOT = 1000
# SEED = 42
# MASK_ARTIFACTS = False        # set True to rerun without mkr/kat/andre/rt/amp features

# GROUPS = [
#     "Persistently incorrect Not-CB",
#     "Persistently incorrect Other-CB",
#     "Persistently correct Not-CB",
#     "Persistently correct Other-CB",
# ]

# # ---------------- REBUILD MISSING OBJECTS ----------------
# assert "persistent_four_df" in globals(), "Run the persistent-group cell first."

# cosine_df = (persistent_four_df[persistent_four_df[GROUP_COL].isin(GROUPS)]
#              .copy().reset_index(drop=True))
# cosine_df[TEXT_COL] = cosine_df[TEXT_COL].fillna("").astype(str)

# # positional indices per group (rows of X_cosine == rows of cosine_df)
# group_indices = {g: np.flatnonzero(cosine_df[GROUP_COL].to_numpy() == g) for g in GROUPS}
# assert all(len(v) > 0 for v in group_indices.values()), "A group is empty."
# all_idx = np.concatenate(list(group_indices.values()))
# assert len(np.unique(all_idx)) == len(all_idx) == len(cosine_df), "Groups overlap."

# # vectorizer = TfidfVectorizer(
# #     analyzer="word", ngram_range=(1, 2), max_features=10000,
# #     sublinear_tf=False, dtype=np.float32
# # )

# vectorizer = TfidfVectorizer(
#     analyzer="word",
#     ngram_range=(1, 2),
#     min_df=3,
#     max_df=0.95,
#     max_features=5000,
#     sublinear_tf=False,
#     dtype=np.float32
# )


# # min_df=3: Excludes features occurring in fewer than three documents.

# # max_df=0.95: Excludes features appearing in more than 95% of documents.

# # max_features=5000: Caps the vocabulary at 5,000 features.


# X_cosine = vectorizer.fit_transform(cosine_df[TEXT_COL])
# terms = np.asarray(vectorizer.get_feature_names_out())

# if MASK_ARTIFACTS:
#     BAD = re.compile(r"\b(mkr|kat|andre|rt|amp)\b")
#     bad_cols = [i for i, t in enumerate(terms) if BAD.search(t)]
#     print("Artifact columns removed:", len(bad_cols), "| 'mkr' in vocab:", "mkr" in vectorizer.vocabulary_)
#     assert len(bad_cols) > 0, "Mask removed nothing - check vocabulary."
#     keep = np.ones(X_cosine.shape[1], dtype=np.float32); keep[bad_cols] = 0
#     X_cosine = X_cosine.multiply(keep).tocsr()

# # explicit L2 normalisation so dot product == cosine
# X_cosine = normalize(X_cosine, norm="l2", axis=1).tocsr()
# print("X_cosine:", X_cosine.shape, "| empty rows:", int((X_cosine.getnnz(axis=1) == 0).sum()))

# # ---------------- BOOTSTRAP ANALYSIS ----------------
# XG = {g: X_cosine[group_indices[g]] for g in GROUPS}
# sq = {g: np.asarray(XG[g].multiply(XG[g]).sum(axis=1)).ravel() for g in GROUPS}  # ||x||^2 per doc

# def between_mean(XA, XB, wA=None, wB=None):
#     wA = np.ones(XA.shape[0]) if wA is None else wA
#     wB = np.ones(XB.shape[0]) if wB is None else wB
#     sA = XA.T.dot(wA); sB = XB.T.dot(wB)
#     return float(sA @ sB) / (wA.sum() * wB.sum())

# def within_mean(X, sqn, w=None):
#     w = np.ones(X.shape[0]) if w is None else w
#     s = X.T.dot(w)
#     num = float(s @ s) - float((w**2 * sqn).sum())   # sum over distinct docs i != j
#     den = w.sum()**2 - (w**2).sum()
#     return num / den

# def boot_ci(fn, sizes, seed):
#     rng = np.random.default_rng(seed)                # own generator -> run-order independent
#     vals = np.empty(N_BOOT)
#     for b in range(N_BOOT):
#         ws = [rng.multinomial(n, np.ones(n) / n).astype(float) for n in sizes]
#         vals[b] = fn(*ws)
#     return np.quantile(vals, [0.025, 0.975])

# # ---- between groups ----
# PAIRS = [
#     ("Persistently incorrect Not-CB",   "Persistently correct Other-CB"),
#     ("Persistently incorrect Other-CB", "Persistently correct Not-CB"),
#     ("Persistently correct Not-CB",     "Persistently incorrect Not-CB"),
#     ("Persistently correct Other-CB",   "Persistently incorrect Other-CB"),
#     ("Persistently correct Not-CB",     "Persistently correct Other-CB"),
#     ("Persistently incorrect Not-CB",   "Persistently incorrect Other-CB"),
# ]
# rows = []
# for k, (a, b) in enumerate(PAIRS):
#     m = between_mean(XG[a], XG[b])
#     lo, hi = boot_ci(lambda wa, wb: between_mean(XG[a], XG[b], wa, wb),
#                      [XG[a].shape[0], XG[b].shape[0]], seed=SEED + k)
#     rows.append({"Group_A": a, "Group_B": b, "Mean_Cosine": m,
#                  "CI95_Lower": lo, "CI95_Upper": hi})
# print("\nBETWEEN-GROUP MEAN PAIRWISE COSINE")
# display(pd.DataFrame(rows).round(5))

# # ---- within groups ----
# rows = []
# for k, g in enumerate(GROUPS):
#     m = within_mean(XG[g], sq[g])
#     lo, hi = boot_ci(lambda w: within_mean(XG[g], sq[g], w),
#                      [XG[g].shape[0]], seed=SEED + 100 + k)
#     rows.append({"Group": g, "Mean_Cosine": m, "CI95_Lower": lo, "CI95_Upper": hi})
# print("\nWITHIN-GROUP MEAN PAIRWISE COSINE")
# display(pd.DataFrame(rows).round(5))

In [21]:
# ============================================================
# LEXICAL SIMILARITY OF PERSISTENT-ERROR GROUPS (TF-IDF cosine)
#
# Statistic : mean pairwise cosine between distinct documents
#             (within-group: i != j ; between-group: all cross pairs)
# Inference : document-level nonparametric bootstrap (multinomial
#             weights), ONE shared replicate for every statistic so
#             differences/contrasts are paired and correctly sized.
# Robustness: (1) main vectorizer, (2) artifact-masked features,
#             (3) alternative vectorizer settings.
#
# Requires  : persistent_four_df with TEXT_COL and GROUP_COL.
# Caveat    : groups were SELECTED by classifiers, so every CI is
#             conditional on the selected groups. Report as
#             descriptive, not as hypothesis tests.
# ============================================================

import re
import platform
import numpy as np
import pandas as pd
import sklearn
from itertools import combinations
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

# ---------------- CONFIG ----------------
TEXT_COL = "lemmatized_tweet"
GROUP_COL = "persistent_group"

N_BOOT = 5000
SEED = 42
ALPHA = 0.05
DEDUPE_WITHIN_GROUP = True          # exact duplicate texts violate bootstrap independence

ARTIFACT_PATTERN = re.compile(r"\b(mkr|kat|andre|rt|amp)\b")

GROUPS = [
    "Persistently incorrect Not-CB",
    "Persistently incorrect Other-CB",
    "Persistently correct Not-CB",
    "Persistently correct Other-CB",
]
SHORT = {
    GROUPS[0]: "Inc Not-CB",
    GROUPS[1]: "Inc Other-CB",
    GROUPS[2]: "Cor Not-CB",
    GROUPS[3]: "Cor Other-CB",
}

CONFIGS = {
    "main": dict(ngram_range=(1, 2), min_df=3, max_df=0.95,
                 max_features=5000, mask=False),
    "masked_artifacts": dict(ngram_range=(1, 2), min_df=3, max_df=0.95,
                             max_features=5000, mask=True),
    "alt_unigram": dict(ngram_range=(1, 1), min_df=2, max_df=0.95,
                        max_features=10000, mask=False),
}

# Pre-specified contrasts. Each: (label, family, positive_stat, negative_stat)
# Family-wise alpha is Bonferroni-adjusted within each family.
def W(g):
    return ("within", g)

def B(a, b):
    a, b = sorted((a, b), key=GROUPS.index)
    return ("between", a, b)

G_IN, G_IO, G_CN, G_CO = GROUPS

CONTRASTS = [
    ("Inc Not-CB closer to Cor Other-CB than to Cor Not-CB",
     "primary", B(G_IN, G_CO), B(G_IN, G_CN)),
    ("Inc Other-CB closer to Cor Not-CB than to Cor Other-CB",
     "primary", B(G_IO, G_CN), B(G_IO, G_CO)),
    ("Within-sim: Inc Not-CB minus Cor Not-CB",
     "primary", W(G_IN), W(G_CN)),
    ("Within-sim: Inc Other-CB minus Cor Other-CB",
     "primary", W(G_IO), W(G_CO)),
    ("Within-sim: Inc Not-CB minus Inc Other-CB",
     "exploratory", W(G_IN), W(G_IO)),
    ("Within-sim: Cor Not-CB minus Cor Other-CB",
     "exploratory", W(G_CN), W(G_CO)),
]

# ---------------- ENVIRONMENT / INPUT CHECKS ----------------
print(f"Python {platform.python_version()} | numpy {np.__version__} | "
      f"pandas {pd.__version__} | sklearn {sklearn.__version__}")
print(f"N_BOOT={N_BOOT}, SEED={SEED}, ALPHA={ALPHA}")

assert "persistent_four_df" in globals(), "Run the persistent-group cell first."
for col in (TEXT_COL, GROUP_COL):
    assert col in persistent_four_df.columns, f"Missing column: {col}"


# ---------------- DATA PREPARATION ----------------
def prepare_base_df():
    df = persistent_four_df[persistent_four_df[GROUP_COL].isin(GROUPS)].copy()
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()
    df = df.reset_index(drop=True)

    report = {"original": df[GROUP_COL].value_counts().reindex(GROUPS)}

    # Texts shared across groups are reported (not removed).
    grp_per_text = df[df[TEXT_COL] != ""].groupby(TEXT_COL)[GROUP_COL].nunique()
    report["texts_in_multiple_groups"] = int((grp_per_text > 1).sum())

    if DEDUPE_WITHIN_GROUP:
        keep = ~(df.duplicated(subset=[TEXT_COL, GROUP_COL], keep="first")
                 & (df[TEXT_COL] != ""))
        report["duplicates_removed"] = (
            df.loc[~keep, GROUP_COL].value_counts().reindex(GROUPS, fill_value=0)
        )
        df = df.loc[keep].reset_index(drop=True)
    else:
        report["duplicates_removed"] = pd.Series(0, index=GROUPS)

    return df, report


base_df, prep_report = prepare_base_df()

print("\nDATA PREPARATION")
print("Texts appearing in more than one group:", prep_report["texts_in_multiple_groups"])
display(pd.DataFrame({
    "Original_N": prep_report["original"],
    "Exact_duplicates_removed": prep_report["duplicates_removed"],
    "After_dedup_N": base_df[GROUP_COL].value_counts().reindex(GROUPS),
}))


# ---------------- FEATURE MATRIX ----------------
def build_matrix(df, cfg):
    vec = TfidfVectorizer(
        analyzer="word",
        ngram_range=cfg["ngram_range"],
        min_df=cfg["min_df"],
        max_df=cfg["max_df"],
        max_features=cfg["max_features"],
        sublinear_tf=False,
        dtype=np.float64,
    )
    X = vec.fit_transform(df[TEXT_COL])
    terms = np.asarray(vec.get_feature_names_out())
    info = {"vocab_size": len(terms), "cols_masked": 0}

    if cfg["mask"]:
        bad = np.array([i for i, t in enumerate(terms)
                        if ARTIFACT_PATTERN.search(t)], dtype=int)
        info["cols_masked"] = len(bad)
        print(f"  Artifact columns removed: {len(bad)} "
              f"(e.g. {list(terms[bad][:6])})")
        assert len(bad) > 0, "Mask removed nothing - check vocabulary."
        keep_cols = np.ones(X.shape[1], dtype=bool)
        keep_cols[bad] = False
        X = X[:, keep_cols].tocsr()

    # Normalise AFTER masking so dot products are true cosines.
    X = normalize(X, norm="l2", axis=1).tocsr()

    empty = np.asarray(X.getnnz(axis=1) == 0).ravel()
    info["empty_by_group"] = (
        df.loc[empty, GROUP_COL].value_counts().reindex(GROUPS, fill_value=0)
    )
    df_kept = df.loc[~empty].reset_index(drop=True)
    X = X[~empty].tocsr()
    return X, df_kept, info


# ---------------- BOOTSTRAP ENGINE ----------------
def _moments(Xg, sqg, w):
    s = np.asarray(Xg.T.dot(w)).ravel()
    return s, float(w.sum()), float(w @ w), float((w * w) @ sqg)


def _all_stats(m):
    """m: group -> (s, W, Q, R). Returns dict stat_key -> value."""
    out = {}
    for g in GROUPS:
        s, Wt, Q, R = m[g]
        # distinct-document pairs only: removes i==j AND copies of the same doc
        out[W(g)] = (s @ s - R) / (Wt * Wt - Q)
    for a, b in combinations(GROUPS, 2):
        out[B(a, b)] = (m[a][0] @ m[b][0]) / (m[a][1] * m[b][1])
    return out


def run_bootstrap(XG, SQ, n_boot, seed):
    keys = list(_all_stats({g: _moments(XG[g], SQ[g], np.ones(XG[g].shape[0]))
                            for g in GROUPS}).keys())
    observed = _all_stats({g: _moments(XG[g], SQ[g], np.ones(XG[g].shape[0]))
                           for g in GROUPS})
    boot = {k: np.empty(n_boot) for k in keys}
    rng = np.random.default_rng(seed)

    for b in range(n_boot):
        m = {}
        for g in GROUPS:
            n = XG[g].shape[0]
            w = rng.multinomial(n, np.full(n, 1.0 / n)).astype(np.float64)
            m[g] = _moments(XG[g], SQ[g], w)
        st = _all_stats(m)
        for k in keys:
            boot[k][b] = st[k]
    return observed, boot


def _label(key):
    if key[0] == "within":
        return "within", SHORT[key[1]], "(within)"
    return "between", SHORT[key[1]], SHORT[key[2]]


# ---------------- ANALYSIS PER CONFIG ----------------
def analyse(config_name, cfg):
    print("\n" + "=" * 78)
    print(f"CONFIG: {config_name}  {cfg}")
    print("=" * 78)

    X, df_kept, info = build_matrix(base_df, cfg)
    print(f"  Vocabulary: {info['vocab_size']} | matrix: {X.shape}")
    print("  Empty vectors dropped by group:")
    display(info["empty_by_group"].rename("Empty_N").to_frame().T)

    labels = df_kept[GROUP_COL].to_numpy()
    idx = {g: np.flatnonzero(labels == g) for g in GROUPS}
    assert all(len(v) > 2 for v in idx.values()), "A group is (almost) empty."
    assert len(np.concatenate(list(idx.values()))) == X.shape[0]

    XG = {g: X[idx[g]] for g in GROUPS}
    SQ = {g: np.asarray(XG[g].multiply(XG[g]).sum(axis=1)).ravel() for g in GROUPS}
    sizes = {g: XG[g].shape[0] for g in GROUPS}
    print("  Group sizes:", {SHORT[g]: n for g, n in sizes.items()})

    observed, boot = run_bootstrap(XG, SQ, N_BOOT, SEED)

    # ---- statistic table ----
    inv_short = {v: k for k, v in SHORT.items()}
    rows, warn_rows = [], []
    for k, obs in observed.items():
        kind, a, b = _label(k)
        v = boot[k]
        lo, hi = np.quantile(v, [ALPHA / 2, 1 - ALPHA / 2])
        inside = lo <= obs <= hi
        if not inside:
            warn_rows.append((kind, a, b, obs, lo, hi))
        rows.append({
            "Type": kind, "Group_A": a, "Group_B": b,
            "N_A": sizes[inv_short[a]],
            "N_B": sizes[inv_short[b]] if kind == "between" else np.nan,
            "Observed": obs, "Boot_Bias": v.mean() - obs, "Boot_SE": v.std(ddof=1),
            "CI95_Lower": lo, "CI95_Upper": hi,
        })
    stats_df = pd.DataFrame(rows)
    print("\n  MEAN PAIRWISE COSINE (95% percentile bootstrap CI)")
    display(stats_df.round(6))
    if warn_rows:
        print("  WARNING: observed value outside its own CI for:", warn_rows)

    # ---- paper-ready matrix ----
    names = [SHORT[g] for g in GROUPS]
    mat = pd.DataFrame("", index=names, columns=names)
    for k, obs in observed.items():
        lo, hi = np.quantile(boot[k], [ALPHA / 2, 1 - ALPHA / 2])
        cell = f"{obs:.5f} [{lo:.5f}, {hi:.5f}]"
        if k[0] == "within":
            mat.loc[SHORT[k[1]], SHORT[k[1]]] = cell
        else:
            mat.loc[SHORT[k[1]], SHORT[k[2]]] = cell
            mat.loc[SHORT[k[2]], SHORT[k[1]]] = cell
    print("\n  SIMILARITY MATRIX (diagonal = within-group, off-diagonal = between-group)")
    display(mat)

    # ---- paired contrasts (same bootstrap replicates) ----
    fam_size = pd.Series([c[1] for c in CONTRASTS]).value_counts().to_dict()
    crow = []
    for label, fam, kp, kn in CONTRASTS:
        d_obs = observed[kp] - observed[kn]
        d_boot = boot[kp] - boot[kn]
        lo, hi = np.quantile(d_boot, [ALPHA / 2, 1 - ALPHA / 2])
        a_adj = ALPHA / fam_size[fam]
        lo_a, hi_a = np.quantile(d_boot, [a_adj / 2, 1 - a_adj / 2])
        crow.append({
            "Config": config_name, "Family": fam, "Contrast": label,
            "Difference": d_obs,
            "CI95_Lower": lo, "CI95_Upper": hi,
            "CI_Excludes_0": bool(lo > 0 or hi < 0),
            "Adj_alpha": a_adj,
            "AdjCI_Lower": lo_a,
            "AdjCI_Upper": hi_a,
            "AdjCI_Excludes_0": bool(lo_a > 0 or hi_a < 0),
            "Boot_Bias": d_boot.mean() - d_obs,
        })
    contrasts_df = pd.DataFrame(crow)
    print("\n  PAIRED CONTRASTS (positive = first term larger)")
    display(contrasts_df.round(6))

    return stats_df, contrasts_df


all_stats, all_contrasts = {}, []
for name, cfg in CONFIGS.items():
    s_df, c_df = analyse(name, cfg)
    all_stats[name] = s_df
    all_contrasts.append(c_df)

contrasts_all = pd.concat(all_contrasts, ignore_index=True)

# ---------------- ROBUSTNESS SUMMARY ----------------
print("\n" + "=" * 78)
print("ROBUSTNESS ACROSS CONFIGURATIONS")
print("=" * 78)

summary_rows = []
for label, fam, _, _ in CONTRASTS:
    sub = contrasts_all[contrasts_all["Contrast"] == label]
    signs = np.sign(sub["Difference"].to_numpy())
    summary_rows.append({
        "Family": fam,
        "Contrast": label,
        **{f"Diff[{r.Config}]": round(r.Difference, 6) for r in sub.itertuples()},
        "Same_sign_all": bool(len(set(signs)) == 1),
        "AdjCI_excludes_0_all": bool(sub["AdjCI_Excludes_0"].all()),
        "AdjCI_excludes_0_any": bool(sub["AdjCI_Excludes_0"].any()),
    })
robustness_df = pd.DataFrame(summary_rows)
display(robustness_df)

print(
    "\nREPORTING GUIDE\n"
    "- Primary claims: only contrasts with AdjCI_excludes_0_all = True AND\n"
    "  Same_sign_all = True are robust to masking and vectorizer choice.\n"
    "- A claim that vanishes under 'masked_artifacts' is a topic artifact\n"
    "  (e.g. mkr), not lexical resemblance between classes.\n"
    "- CIs are conditional on classifier-selected groups: describe them as\n"
    "  'bootstrap intervals', not as significance tests.\n"
    "- Absolute cosine values depend on the vectorizer; interpret contrasts."
)

Python 3.13.16 | numpy 2.1.3 | pandas 2.2.3 | sklearn 1.6.1
N_BOOT=5000, SEED=42, ALPHA=0.05

DATA PREPARATION
Texts appearing in more than one group: 12


,Original_N,Exact_duplicates_removed,After_dedup_N
persistent_group,,,
Persistently incorrect Not-CB,1432,5,1427
Persistently incorrect Other-CB,1671,8,1663
Persistently correct Not-CB,3185,31,3154
Persistently correct Other-CB,2163,22,2141



CONFIG: main  {'ngram_range': (1, 2), 'min_df': 3, 'max_df': 0.95, 'max_features': 5000, 'mask': False}
  Vocabulary: 4630 | matrix: (8323, 4630)
  Empty vectors dropped by group:


persistent_group,Persistently incorrect Not-CB,Persistently incorrect Other-CB,Persistently correct Not-CB,Persistently correct Other-CB
Empty_N,4,33,25,0


  Group sizes: {'Inc Not-CB': 1423, 'Inc Other-CB': 1630, 'Cor Not-CB': 3129, 'Cor Other-CB': 2141}

  MEAN PAIRWISE COSINE (95% percentile bootstrap CI)


...


  PAIRED CONTRASTS (positive = first term larger)


,Config,Family,Contrast,Difference,CI95_Lower,CI95_Upper,CI_Excludes_0,Adj_alpha,AdjCI_Lower,AdjCI_Upper,AdjCI_Excludes_0,Boot_Bias
0,alt_unigram,primary,Inc Not-CB closer to Cor Other-CB than to Cor ...,0.004091,0.003762,0.004433,True,0.0125,0.003669,0.004530,True,-0.000001
1,alt_unigram,primary,Inc Other-CB closer to Cor Not-CB than to Cor ...,0.000276,-0.000066,0.000613,False,0.0125,-0.000156,0.000721,False,-0.000002
2,alt_unigram,primary,Within-sim: Inc Not-CB minus Cor Not-CB,-0.004111,-0.004956,-0.003262,True,0.0125,-0.005191,-0.002996,True,-0.000002
3,alt_unigram,primary,Within-sim: Inc Other-CB minus Cor Other-CB,-0.007500,-0.008237,-0.006781,True,0.0125,-0.008413,-0.006591,True,0.000000
4,alt_unigram,exploratory,Within-sim: Inc Not-CB minus Inc Other-CB,0.002617,0.002021,0.003220,True,0.0250,0.001924,0.003305,True,-0.000002
5,alt_unigram,exploratory,Within-sim: Cor Not-CB minus Cor Other-CB,-0.000772,-0.001714,0.000172,False,0.0250,-0.001862,0.000299,False,0.000001



ROBUSTNESS ACROSS CONFIGURATIONS


,Family,Contrast,Diff[main],Diff[masked_artifacts],Diff[alt_unigram],Same_sign_all,AdjCI_excludes_0_all,AdjCI_excludes_0_any
0,primary,Inc Not-CB closer to Cor Other-CB than to Cor ...,0.003676,0.003492,0.004091,True,True,True
1,primary,Inc Other-CB closer to Cor Not-CB than to Cor ...,0.000350,-0.000399,0.000276,False,False,True
2,primary,Within-sim: Inc Not-CB minus Cor Not-CB,-0.003316,0.002012,-0.004111,False,True,True
3,primary,Within-sim: Inc Other-CB minus Cor Other-CB,-0.006186,-0.006073,-0.007500,True,True,True
4,exploratory,Within-sim: Inc Not-CB minus Inc Other-CB,0.002395,0.002547,0.002617,True,True,True
5,exploratory,Within-sim: Cor Not-CB minus Cor Other-CB,-0.000475,-0.005539,-0.000772,True,False,True



REPORTING GUIDE
- Primary claims: only contrasts with AdjCI_excludes_0_all = True AND
  Same_sign_all = True are robust to masking and vectorizer choice.
- A claim that vanishes under 'masked_artifacts' is a topic artifact
  (e.g. mkr), not lexical resemblance between classes.
- CIs are conditional on classifier-selected groups: describe them as
  'bootstrap intervals', not as significance tests.
- Absolute cosine values depend on the vectorizer; interpret contrasts.


In [22]:
# # ============================================================
# # CELL 6 (CORRECTED) — STRENGTHENING THE COSINE ANALYSIS
# # Run Cell 5 first (needs X_cosine, vectorizer, group_indices,
# # GROUPS, cosine_df, TEXT_COL).
# #
# # What changed vs the first version:
# #   1) Centroid matrix: POINT ESTIMATES ONLY (old bootstrap CIs were invalid)
# #   2) Permutation test: correct null (inc docs vs random subset of
# #      their OWN true class) + document-level bootstrap CI
# #   3) Sensitivity run without mkr / kat / andre / rt / amp
# #   4) p-values reported as "< 0.0001" when at the minimum possible
# #   5) Duplicate / retweet / mkr check per group
# #   6) Top-terms and sparsity tables unchanged
# # ============================================================

# import numpy as np
# import pandas as pd
# import scipy.sparse as sp
# from IPython.display import display
# from sklearn.metrics.pairwise import cosine_similarity

# RANDOM_STATE = 42
# N_PERM = 10_000
# N_BOOT = 1000
# TOP_N = 25
# rng = np.random.default_rng(RANDOM_STATE)

# SHORT = {
#     "Persistently incorrect Not-CB":   "Inc Not-CB",
#     "Persistently incorrect Other-CB": "Inc Other-CB",
#     "Persistently correct Not-CB":     "Cor Not-CB",
#     "Persistently correct Other-CB":   "Cor Other-CB",
# }
# short_names = [SHORT[g] for g in GROUPS]

# XG = {g: X_cosine[group_indices[g]].tocsr() for g in GROUPS}
# terms = np.array(vectorizer.get_feature_names_out())
# vocab = vectorizer.vocabulary_


# def centroid(X, w=None):
#     """(Weighted) mean TF-IDF vector of a group."""
#     w = np.ones(X.shape[0]) if w is None else w
#     return np.asarray(X.T.dot(w)).ravel() / w.sum()


# def boot_weights(n):
#     """Document-level bootstrap as multinomial counts."""
#     return rng.multinomial(n, np.ones(n) / n).astype(float)


# def fmt_p(p, n_perm=N_PERM):
#     floor = 1 / (n_perm + 1)
#     return f"< {floor:.4f}" if p <= floor + 1e-12 else f"{p:.4f}"


# def renorm_without(X, bad_terms):
#     """Zero out chosen terms, then L2-renormalise every row."""
#     keep = np.ones(X.shape[1], dtype=np.float32)
#     for t in bad_terms:
#         if t in vocab:
#             keep[vocab[t]] = 0
#     Xn = X.multiply(keep).tocsr()
#     n = np.sqrt(np.asarray(Xn.multiply(Xn).sum(axis=1)).ravel())
#     n[n == 0] = 1
#     return (sp.diags(1 / n) @ Xn).tocsr()


# # ============================================================
# # 1) CENTROID COSINE MATRIX (point estimates, descriptive only)
# # ============================================================
# def centroid_matrix(X):
#     C = np.vstack([centroid(X[group_indices[g]]) for g in GROUPS])
#     return pd.DataFrame(cosine_similarity(C),
#                         index=short_names, columns=short_names).round(3)


# print("1) CENTROID COSINE MATRIX (all terms)")
# display(centroid_matrix(X_cosine))


# # ============================================================
# # 2) PERMUTATION TEST (corrected null)
# # Question: is the persistently-incorrect group more similar to the
# # OTHER class's correct tweets than a random same-size sample of its
# # OWN true class would be?
# #   pool = inc + cor of the SAME true class
# #   statistic = mean cos(inc docs, centroid of other-class correct)
# #               - mean cos(rest of pool, same centroid)
# #   null = randomly relabel which pool docs are "inc"
# # ============================================================
# def class_perm_test(X, inc, cor, other, n_perm=N_PERM, n_boot=N_BOOT):
#     i_inc, i_cor, i_oth = (group_indices[inc], group_indices[cor],
#                            group_indices[other])
#     X_inc, X_cor, X_oth = X[i_inc], X[i_cor], X[i_oth]

#     c_other = centroid(X_oth)
#     s_inc = np.asarray(X_inc.dot(c_other)).ravel()
#     s_cor = np.asarray(X_cor.dot(c_other)).ravel()
#     observed = s_inc.mean() - s_cor.mean()

#     pooled = np.concatenate([s_inc, s_cor])
#     n_inc = len(s_inc)
#     null = np.empty(n_perm)
#     for i in range(n_perm):
#         perm = rng.permutation(len(pooled))
#         null[i] = pooled[perm[:n_inc]].mean() - pooled[perm[n_inc:]].mean()
#     p_one = (1 + np.sum(null >= observed)) / (n_perm + 1)

#     # document-level bootstrap CI for the observed difference
#     boots = np.empty(n_boot)
#     for b in range(n_boot):
#         c = centroid(X_oth, boot_weights(X_oth.shape[0]))
#         a = np.asarray(X_inc.dot(c)).ravel()
#         r = np.asarray(X_cor.dot(c)).ravel()
#         boots[b] = (np.average(a, weights=boot_weights(len(a)))
#                     - np.average(r, weights=boot_weights(len(r))))
#     ci_lo, ci_hi = np.quantile(boots, [0.025, 0.975])

#     return {
#         "Incorrect_Group": SHORT[inc],
#         "Compared_To": f"{SHORT[cor]} (own class) vs {SHORT[other]}",
#         "Mean_Cos_Inc": s_inc.mean(),
#         "Mean_Cos_Own_Class_Correct": s_cor.mean(),
#         "Difference": observed,
#         "Boot_CI95_Lower": ci_lo,
#         "Boot_CI95_Upper": ci_hi,
#         "p_one_sided": fmt_p(p_one),
#     }


# TESTS = [
#     ("Persistently incorrect Not-CB",
#      "Persistently correct Not-CB",
#      "Persistently correct Other-CB"),
#     ("Persistently incorrect Other-CB",
#      "Persistently correct Other-CB",
#      "Persistently correct Not-CB"),
# ]

# X_clean = renorm_without(
#     X_cosine, ["mkr", "kat", "andre", "kat andre", "rt", "amp"]
# )

# print("\n2) PERMUTATION TEST — positive difference = incorrect tweets are "
#       "closer to the OTHER class than normal tweets of their own class")
# for label, X in [("ALL TERMS", X_cosine),
#                  ("WITHOUT mkr / kat / andre / rt / amp", X_clean)]:
#     rows = [class_perm_test(X, inc, cor, oth) for inc, cor, oth in TESTS]
#     print(f"\n--- {label} ---")
#     display(pd.DataFrame(rows).round(5))

# print("\nCentroid matrix WITHOUT mkr / kat / andre / rt / amp "
#       "(does the pattern survive?)")
# display(centroid_matrix(X_clean))


# # ============================================================
# # 3) TOP TERMS (all-terms matrix)
# # ============================================================
# cent = {g: centroid(XG[g]) for g in GROUPS}
# doc_freq = {g: np.asarray((XG[g] > 0).mean(axis=0)).ravel() for g in GROUPS}

# top_tbl = {}
# for g in GROUPS:
#     idx = np.argsort(cent[g])[::-1][:TOP_N]
#     top_tbl[SHORT[g]] = [f"{terms[i]} ({100 * doc_freq[g][i]:.1f}%)"
#                          for i in idx]
# print("\n3a) TOP TERMS PER GROUP (% of docs containing the term)")
# display(pd.DataFrame(top_tbl, index=range(1, TOP_N + 1)))


# def overlap_terms(a, b, top_n=TOP_N):
#     contrib = cent[a] * cent[b]       # exact share of mean pairwise cosine
#     total = contrib.sum()
#     idx = np.argsort(contrib)[::-1][:top_n]
#     out = pd.DataFrame({
#         "Term": terms[idx],
#         "Share_of_Overlap_%": 100 * contrib[idx] / total,
#         f"DocFreq_{SHORT[a]}_%": 100 * doc_freq[a][idx],
#         f"DocFreq_{SHORT[b]}_%": 100 * doc_freq[b][idx],
#     })
#     out["Cumulative_%"] = out["Share_of_Overlap_%"].cumsum()
#     return out, total


# for a, b in [
#     ("Persistently incorrect Not-CB",   "Persistently correct Other-CB"),
#     ("Persistently incorrect Not-CB",   "Persistently correct Not-CB"),
#     ("Persistently incorrect Other-CB", "Persistently correct Not-CB"),
# ]:
#     tbl, total = overlap_terms(a, b)
#     print(f"\n3b) OVERLAP DRIVERS: {SHORT[a]} vs {SHORT[b]} "
#           f"(mean cosine = {total:.5f})")
#     display(tbl.round(3))


# # ============================================================
# # 4) SPARSITY: FRACTION OF NONZERO PAIRS (exact)
# # ============================================================
# def sparsity_row(a, b=None):
#     XA = XG[a]
#     if b is None:                                  # within group
#         P = (XA @ XA.T).tocsr()
#         P.eliminate_zeros()
#         diag = P.diagonal()
#         n = XA.shape[0]
#         total_pairs = n * (n - 1)
#         nnz = P.nnz - np.count_nonzero(diag)
#         total_sum = P.sum() - diag.sum()
#         label_b = "(within)"
#     else:
#         P = (XA @ XG[b].T).tocsr()
#         P.eliminate_zeros()
#         total_pairs = XA.shape[0] * XG[b].shape[0]
#         nnz = P.nnz
#         total_sum = P.sum()
#         label_b = SHORT[b]
#     return {
#         "Group_A": SHORT[a],
#         "Group_B": label_b,
#         "Total_Pairs": total_pairs,
#         "Nonzero_Pairs": int(nnz),
#         "Fraction_Nonzero_%": 100 * nnz / total_pairs,
#         "Mean_Cosine_All_Pairs": total_sum / total_pairs,
#         "Mean_Cosine_Nonzero_Only": total_sum / nnz if nnz else np.nan,
#     }


# sparsity_rows = [sparsity_row(g) for g in GROUPS]
# for a, b in [
#     ("Persistently incorrect Not-CB",   "Persistently correct Other-CB"),
#     ("Persistently incorrect Other-CB", "Persistently correct Not-CB"),
#     ("Persistently correct Not-CB",     "Persistently incorrect Not-CB"),
#     ("Persistently correct Other-CB",   "Persistently incorrect Other-CB"),
#     ("Persistently correct Not-CB",     "Persistently correct Other-CB"),
#     ("Persistently incorrect Not-CB",   "Persistently incorrect Other-CB"),
# ]:
#     sparsity_rows.append(sparsity_row(a, b))

# print("\n4) SPARSITY OF PAIRWISE COMPARISONS")
# display(pd.DataFrame(sparsity_rows).round(5))


# # ============================================================
# # 5) DUPLICATES, RETWEETS AND "mkr" CHECK PER GROUP
# # ============================================================
# texts = cosine_df[TEXT_COL]
# dup_rows = []
# for g in GROUPS:
#     t = texts.iloc[group_indices[g]]
#     dup_rows.append({
#         "Group": SHORT[g],
#         "Docs": len(t),
#         "Unique_Texts": t.nunique(),
#         "Duplicate_%": 100 * (1 - t.nunique() / len(t)),
#         "Contains_rt_%": 100 * t.str.contains(r"\brt\b").mean(),
#         "Contains_mkr_%": 100 * t.str.contains(r"\bmkr\b").mean(),
#     })
# print("\n5) DUPLICATES / RETWEETS / mkr")
# display(pd.DataFrame(dup_rows).round(2))

# print("\nCell 6 (corrected) completed.")

In [23]:
# ============================================================
# CELL 6 (SELF-CONTAINED, CORRECTED) — LEXICAL SIMILARITY AND
# OPPOSITE-VS-OWN MARGIN ANALYSIS FOR PERSISTENT-ERROR GROUPS
#
# Requires only: persistent_four_df  (TEXT_COL + GROUP_COL)
#
# What is reported (all DESCRIPTIVE, conditional on the
# classifier-selected groups; no p-values):
#   0. Duplicate / retweet / mkr audit (raw, before de-duplication)
#   1. Centroid cosine matrix:
#        - naive (size-dependent, for comparison with earlier runs)
#        - bias-corrected (self-pairs removed) with bootstrap CIs
#      + paired contrasts (Bonferroni-adjusted)
#   2. Opposite-vs-own centroid margin of the incorrect groups
#        (mean, median, % closer to opposite, inc-vs-cor difference)
#   3. Top terms and lexical contributions        (main config only)
#   4. Sparsity of pairwise comparisons           (main config only)
#   5. Robustness summary across three configurations
# ============================================================

import re
import platform
import numpy as np
import pandas as pd
import sklearn
from itertools import combinations
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

# ---------------- CONFIG ----------------
TEXT_COL = "lemmatized_tweet"
GROUP_COL = "persistent_group"

N_BOOT = 2000
SEED = 42
ALPHA = 0.05
TOP_N = 25
BLOCK_SIZE = 256
DEDUPE_WITHIN_GROUP = True      # exact duplicates violate bootstrap independence

ARTIFACT_PATTERN = re.compile(r"\b(mkr|kat|andre|rt|amp)\b")

GROUPS = [
    "Persistently incorrect Not-CB",
    "Persistently incorrect Other-CB",
    "Persistently correct Not-CB",
    "Persistently correct Other-CB",
]
SHORT = {
    GROUPS[0]: "Inc Not-CB",
    GROUPS[1]: "Inc Other-CB",
    GROUPS[2]: "Cor Not-CB",
    GROUPS[3]: "Cor Other-CB",
}
G_IN, G_IO, G_CN, G_CO = GROUPS

CONFIGS = {
    "main": dict(ngram_range=(1, 2), min_df=3, max_df=0.95,
                 max_features=5000, mask=False),
    "masked_artifacts": dict(ngram_range=(1, 2), min_df=3, max_df=0.95,
                             max_features=5000, mask=True),
    "alt_unigram": dict(ngram_range=(1, 1), min_df=2, max_df=0.95,
                        max_features=10000, mask=False),
}

# incorrect group, its own-class correct group, the opposite-class correct group
TESTS = [
    (G_IN, G_CN, G_CO),
    (G_IO, G_CO, G_CN),
]

print(f"Python {platform.python_version()} | numpy {np.__version__} | "
      f"pandas {pd.__version__} | sklearn {sklearn.__version__}")
print(f"N_BOOT={N_BOOT}, SEED={SEED}, ALPHA={ALPHA}")

assert "persistent_four_df" in globals(), "Run the persistent-group cell first."
for col in (TEXT_COL, GROUP_COL):
    assert col in persistent_four_df.columns, f"Missing column: {col}"


# ============================================================
# 0. DATA PREPARATION AND AUDIT
# ============================================================
raw_df = persistent_four_df[persistent_four_df[GROUP_COL].isin(GROUPS)].copy()
raw_df[TEXT_COL] = raw_df[TEXT_COL].fillna("").astype(str).str.strip()
raw_df = raw_df.reset_index(drop=True)


def audit_table(df):
    rows = []
    for g in GROUPS:
        t = df.loc[df[GROUP_COL] == g, TEXT_COL]
        n = len(t)
        rows.append({
            "Group": SHORT[g],
            "Documents": n,
            "Unique_Texts": t.nunique(),
            "Duplicate_%": 100 * (1 - t.nunique() / n) if n else np.nan,
            "Contains_rt_%": 100 * t.str.contains(r"\brt\b", case=False, regex=True).mean(),
            "Contains_mkr_%": 100 * t.str.contains(r"\bmkr\b", case=False, regex=True).mean(),
        })
    return pd.DataFrame(rows)


print("\n0. AUDIT (raw groups, before de-duplication)")
display(audit_table(raw_df).round(3))

nonempty = raw_df[raw_df[TEXT_COL] != ""]
n_shared = int((nonempty.groupby(TEXT_COL)[GROUP_COL].nunique() > 1).sum())
print("Texts appearing in more than one group (label inconsistency candidates):", n_shared)

if DEDUPE_WITHIN_GROUP:
    dup_mask = (raw_df.duplicated(subset=[TEXT_COL, GROUP_COL], keep="first")
                & (raw_df[TEXT_COL] != ""))
    removed = raw_df.loc[dup_mask, GROUP_COL].value_counts().reindex(GROUPS, fill_value=0)
    base_df = raw_df.loc[~dup_mask].reset_index(drop=True)
else:
    removed = pd.Series(0, index=GROUPS)
    base_df = raw_df.copy()

display(pd.DataFrame({
    "Original_N": raw_df[GROUP_COL].value_counts().reindex(GROUPS),
    "Duplicates_removed": removed,
    "After_dedup_N": base_df[GROUP_COL].value_counts().reindex(GROUPS),
}))


# ============================================================
# FEATURE MATRIX
# ============================================================
def build_matrix(df, cfg):
    vec = TfidfVectorizer(
        analyzer="word", ngram_range=cfg["ngram_range"],
        min_df=cfg["min_df"], max_df=cfg["max_df"],
        max_features=cfg["max_features"], sublinear_tf=False, dtype=np.float64,
    )
    X = vec.fit_transform(df[TEXT_COL])
    terms = np.asarray(vec.get_feature_names_out())
    info = {"vocab_size": len(terms), "cols_masked": 0}

    if cfg["mask"]:
        bad = np.array([i for i, t in enumerate(terms) if ARTIFACT_PATTERN.search(t)],
                       dtype=int)
        info["cols_masked"] = len(bad)
        print(f"  Artifact columns removed: {len(bad)} (e.g. {list(terms[bad][:6])})")
        assert len(bad) > 0, "Mask removed nothing - check the vocabulary."
        keep = np.ones(X.shape[1], dtype=bool)
        keep[bad] = False
        X = X[:, keep].tocsr()
        terms = terms[keep]

    X = normalize(X, norm="l2", axis=1).tocsr()          # normalise AFTER masking
    empty = np.asarray(X.getnnz(axis=1) == 0).ravel()
    info["empty_by_group"] = (df.loc[empty, GROUP_COL]
                              .value_counts().reindex(GROUPS, fill_value=0))
    return X[~empty].tocsr(), df.loc[~empty].reset_index(drop=True), terms, info


# ============================================================
# BOOTSTRAP HELPERS (all use their own seeded generator)
# ============================================================
def _moments(Xg, sqg, w):
    s = np.asarray(Xg.T.dot(w)).ravel()
    return s, float(w.sum()), float(w @ w), float((w * w) @ sqg)


def _within(m):
    s, Wt, Q, R = m
    return (s @ s - R) / (Wt * Wt - Q)          # distinct-document pairs only


def _between(ma, mb):
    return (ma[0] @ mb[0]) / (ma[1] * mb[1])


def _corrected_cosines(m):
    """Bias-corrected centroid cosine: B_ab / sqrt(W_a * W_b)."""
    wi = {g: _within(m[g]) for g in GROUPS}
    out = {}
    for a, b in combinations(GROUPS, 2):
        d = wi[a] * wi[b]
        out[(a, b)] = _between(m[a], m[b]) / np.sqrt(d) if d > 0 else np.nan
    return out


def _unit(c):
    n = np.linalg.norm(c)
    return c / n if n > 0 else c


def _wmedian(values, w):
    order = np.argsort(values, kind="stable")
    cw = np.cumsum(w[order])
    return float(values[order][np.searchsorted(cw, 0.5 * cw[-1])])


def _ci(v, alpha=ALPHA):
    v = v[np.isfinite(v)]
    return np.quantile(v, [alpha / 2, 1 - alpha / 2])


def corrected_cosine_bootstrap(XG, SQ, seed):
    ones = {g: np.ones(XG[g].shape[0]) for g in GROUPS}
    obs = _corrected_cosines({g: _moments(XG[g], SQ[g], ones[g]) for g in GROUPS})
    pairs = list(obs.keys())
    boot = {k: np.empty(N_BOOT) for k in pairs}
    rng = np.random.default_rng(seed)
    for b in range(N_BOOT):
        m = {}
        for g in GROUPS:
            n = XG[g].shape[0]
            m[g] = _moments(XG[g], SQ[g],
                            rng.multinomial(n, np.full(n, 1.0 / n)).astype(float))
        st = _corrected_cosines(m)
        for k in pairs:
            boot[k][b] = st[k]
    return obs, boot


def margin_bootstrap(X_inc, X_own, X_opp, seed):
    """
    margin_i = cos(doc_i, opposite-class correct centroid)
             - cos(doc_i, own-class correct centroid)      (incorrect docs only)
    Bootstrap resamples incorrect docs AND both reference groups, so the
    uncertainty in the centroids is propagated.
    """
    def centroid(X, w):
        return np.asarray(X.T.dot(w)).ravel() / w.sum()

    n_i, n_o, n_p = X_inc.shape[0], X_own.shape[0], X_opp.shape[0]
    c_own = _unit(centroid(X_own, np.ones(n_o)))
    c_opp = _unit(centroid(X_opp, np.ones(n_p)))

    s_opp = X_inc.dot(c_opp)
    margins = s_opp - X_inc.dot(c_own)
    w1 = np.ones(n_i)
    obs = {
        "Mean_Margin": float(margins.mean()),
        "Median_Margin": _wmedian(margins, w1),
        "Pct_Closer_to_Opposite": 100 * float(np.mean(margins > 0)),
        "Pct_Closer_to_Own": 100 * float(np.mean(margins < 0)),
        "Diff_Inc_minus_Cor_cos_to_Opposite":
            float(s_opp.mean() - X_own.dot(c_opp).mean()),
    }
    pct_tied = 100 * float(np.mean(margins == 0))

    boot = {k: np.empty(N_BOOT) for k in obs}
    rng = np.random.default_rng(seed)
    for b in range(N_BOOT):
        wi = rng.multinomial(n_i, np.full(n_i, 1.0 / n_i)).astype(float)
        wo = rng.multinomial(n_o, np.full(n_o, 1.0 / n_o)).astype(float)
        wp = rng.multinomial(n_p, np.full(n_p, 1.0 / n_p)).astype(float)
        co_b = _unit(centroid(X_own, wo))
        cp_b = _unit(centroid(X_opp, wp))
        sp_b = X_inc.dot(cp_b)
        m_b = sp_b - X_inc.dot(co_b)
        sw = wi.sum()
        boot["Mean_Margin"][b] = (wi @ m_b) / sw
        boot["Median_Margin"][b] = _wmedian(m_b, wi)
        boot["Pct_Closer_to_Opposite"][b] = 100 * (wi @ (m_b > 0)) / sw
        boot["Pct_Closer_to_Own"][b] = 100 * (wi @ (m_b < 0)) / sw
        boot["Diff_Inc_minus_Cor_cos_to_Opposite"][b] = (
            (wi @ sp_b) / sw - (wo @ X_own.dot(cp_b)) / wo.sum()
        )
    return obs, boot, pct_tied


# ============================================================
# ANALYSIS PER CONFIGURATION
# ============================================================
def analyse(cfg_idx, name, cfg):
    print("\n" + "=" * 78)
    print(f"CONFIG: {name}  {cfg}")
    print("=" * 78)

    X, df_kept, terms, info = build_matrix(base_df, cfg)
    print(f"  Vocabulary: {info['vocab_size']} | matrix: {X.shape}")
    print("  Empty vectors dropped by group:")
    display(info["empty_by_group"].rename("Empty_N").to_frame().T)

    labels = df_kept[GROUP_COL].to_numpy()
    idx = {g: np.flatnonzero(labels == g) for g in GROUPS}
    assert all(len(v) > 2 for v in idx.values()), "A group is (almost) empty."
    XG = {g: X[idx[g]] for g in GROUPS}
    SQ = {g: np.asarray(XG[g].multiply(XG[g]).sum(axis=1)).ravel() for g in GROUPS}
    print("  Group sizes:", {SHORT[g]: XG[g].shape[0] for g in GROUPS})

    names = [SHORT[g] for g in GROUPS]
    out = {"X": X, "XG": XG, "SQ": SQ, "terms": terms, "idx": idx, "df": df_kept}

    # ---------- 1a. naive centroid cosine (point estimates) ----------
    cents = np.vstack([np.asarray(XG[g].mean(axis=0)).ravel() for g in GROUPS])
    naive = pd.DataFrame(cosine_similarity(cents), index=names, columns=names)
    print("\n  1a. NAIVE CENTROID COSINE (point estimates; shrinks for small groups,")
    print("      so values are NOT comparable across groups of different size)")
    display(naive.round(4))

    # ---------- 1b. bias-corrected centroid cosine + CIs ----------
    obs, boot = corrected_cosine_bootstrap(XG, SQ, seed=[SEED, cfg_idx, 1])
    mat = pd.DataFrame("1", index=names, columns=names)
    warn = []
    for (a, b), v in obs.items():
        lo, hi = _ci(boot[(a, b)])
        if not (lo <= v <= hi):
            warn.append((SHORT[a], SHORT[b], v, lo, hi))
        cell = f"{v:.4f} [{lo:.4f}, {hi:.4f}]"
        mat.loc[SHORT[a], SHORT[b]] = cell
        mat.loc[SHORT[b], SHORT[a]] = cell
    print("\n  1b. BIAS-CORRECTED CENTROID COSINE  B_ab / sqrt(W_a W_b)  [95% bootstrap CI]")
    print("      (self-pairs removed; diagonal = 1 by definition)")
    display(mat)
    if warn:
        print("  WARNING: observed outside own CI:", warn)

    # ---------- 1c. paired contrasts (Bonferroni, family of 2) ----------
    def key(a, b):
        a, b = sorted((a, b), key=GROUPS.index)
        return (a, b)

    CONTR = [
        ("Inc Not-CB closer to Cor Other-CB than to Cor Not-CB",
         key(G_IN, G_CO), key(G_IN, G_CN)),
        ("Inc Other-CB closer to Cor Not-CB than to Cor Other-CB",
         key(G_IO, G_CN), key(G_IO, G_CO)),
    ]
    a_adj = ALPHA / len(CONTR)
    crow = []
    for label, kp, kn in CONTR:
        d_obs = obs[kp] - obs[kn]
        d_boot = boot[kp] - boot[kn]
        lo, hi = _ci(d_boot)
        lo_a, hi_a = _ci(d_boot, a_adj)
        crow.append({
            "Config": name, "Contrast": label, "Difference": d_obs,
            "CI95_Lower": lo, "CI95_Upper": hi,
            "AdjCI_Lower": lo_a, "AdjCI_Upper": hi_a,
            "AdjCI_Excludes_0": bool(lo_a > 0 or hi_a < 0),
            "Boot_Bias": float(np.nanmean(d_boot) - d_obs),
        })
    contrasts_df = pd.DataFrame(crow)
    print("\n  1c. PAIRED CONTRASTS OF CORRECTED COSINES (positive = first term larger)")
    display(contrasts_df.round(5))

    # ---------- 2. margin analysis ----------
    mrows, mlong = [], []
    for k, (inc, own, opp) in enumerate(TESTS):
        o, bt, tied = margin_bootstrap(XG[inc], XG[own], XG[opp],
                                       seed=[SEED, cfg_idx, 2, k])
        row = {"Config": name, "Incorrect_Group": SHORT[inc],
               "N_Documents": XG[inc].shape[0], "Pct_Tied": tied}
        for stat, v in o.items():
            lo, hi = _ci(bt[stat])
            row[stat] = v
            row[stat + "_CI_Lower"] = lo
            row[stat + "_CI_Upper"] = hi
            row[stat + "_Boot_Bias"] = float(bt[stat].mean() - v)
        mrows.append(row)
    margin_df = pd.DataFrame(mrows)
    show_cols = ["Incorrect_Group", "N_Documents"]
    for stat in ["Mean_Margin", "Median_Margin", "Pct_Closer_to_Opposite",
                 "Pct_Closer_to_Own", "Diff_Inc_minus_Cor_cos_to_Opposite"]:
        show_cols += [stat, stat + "_CI_Lower", stat + "_CI_Upper"]
    print("\n  2. OPPOSITE-VS-OWN CENTROID MARGIN OF INCORRECT GROUPS")
    print("     margin > 0  => document closer to the opposite-class correct centroid")
    display(margin_df[show_cols].round(5))

    out.update(naive=naive, contrasts=contrasts_df, margin=margin_df,
               corrected_obs=obs)
    return out


results = {}
for i, (name, cfg) in enumerate(CONFIGS.items()):
    results[name] = analyse(i, name, cfg)


# ============================================================
# 3. TOP TERMS AND LEXICAL CONTRIBUTIONS (main config, descriptive)
# ============================================================
R = results["main"]
XG, terms = R["XG"], R["terms"]

cent = {g: np.asarray(XG[g].mean(axis=0)).ravel() for g in GROUPS}
doc_freq = {g: np.asarray((XG[g] > 0).mean(axis=0)).ravel() for g in GROUPS}

top_tbl = {}
for g in GROUPS:
    idx_top = np.argsort(cent[g])[::-1][:TOP_N]
    top_tbl[SHORT[g]] = [f"{terms[i]} ({100 * doc_freq[g][i]:.1f}%)" for i in idx_top]
print("\n3A. TOP TERMS PER GROUP (mean TF-IDF; % of documents containing the term)")
display(pd.DataFrame(top_tbl, index=range(1, TOP_N + 1)))


def overlap_terms(a, b, top_n=TOP_N):
    """Per-term share of the mean cross-group cosine (a != b, so no self-pairs)."""
    contrib = cent[a] * cent[b]
    total = float(contrib.sum())
    idx_top = np.argsort(contrib)[::-1][:top_n]
    shares = 100 * contrib[idx_top] / total if total > 0 else np.zeros(len(idx_top))
    out = pd.DataFrame({
        "Term": terms[idx_top],
        "Contribution_to_Mean_Cosine": contrib[idx_top],
        "Share_%": shares,
        f"DocFreq_{SHORT[a]}_%": 100 * doc_freq[a][idx_top],
        f"DocFreq_{SHORT[b]}_%": 100 * doc_freq[b][idx_top],
    })
    out["Cumulative_Share_%"] = out["Share_%"].cumsum()
    return out, total


for a, b in [(G_IN, G_CO), (G_IN, G_CN), (G_IO, G_CN)]:
    tbl, total = overlap_terms(a, b)
    print(f"\n3B. LEXICAL CONTRIBUTIONS: {SHORT[a]} vs {SHORT[b]} "
          f"(mean pairwise cosine = {total:.6f})")
    display(tbl.round(5))


# ============================================================
# 4. SPARSITY OF PAIRWISE COMPARISONS (main config)
# ============================================================
SQ = R["SQ"]


def sparsity_row(a, b=None):
    XA = XG[a]
    XB = XA if b is None else XG[b]
    n_a, n_b = XA.shape[0], XB.shape[0]
    total_pairs = n_a * (n_a - 1) if b is None else n_a * n_b
    nnz_total, cos_sum = 0, 0.0
    for start in range(0, n_a, BLOCK_SIZE):
        stop = min(start + BLOCK_SIZE, n_a)
        P = (XA[start:stop] @ XB.T).tocsr()
        P.eliminate_zeros()
        if b is None:                      # remove the n self-pairs (cos = ||x||^2, nonzero)
            nnz_total += P.nnz - (stop - start)
            cos_sum += P.sum() - SQ[a][start:stop].sum()
        else:
            nnz_total += P.nnz
            cos_sum += P.sum()
    return {
        "Group_A": SHORT[a], "Group_B": "(within)" if b is None else SHORT[b],
        "Total_Pairs": int(total_pairs), "Nonzero_Pairs": int(nnz_total),
        "Fraction_Nonzero_%": 100 * nnz_total / total_pairs,
        "Mean_Cosine_All_Pairs": cos_sum / total_pairs,
        "Mean_Cosine_Nonzero_Only": cos_sum / nnz_total if nnz_total else np.nan,
    }


sp_rows = [sparsity_row(g) for g in GROUPS]
sp_rows += [sparsity_row(a, b) for a, b in combinations(GROUPS, 2)]
print("\n4. SPARSITY OF PAIRWISE COMPARISONS (main config)")
display(pd.DataFrame(sp_rows).round(6))


# ============================================================
# 5. ROBUSTNESS SUMMARY ACROSS CONFIGURATIONS
# ============================================================
print("\n" + "=" * 78)
print("5. ROBUSTNESS SUMMARY")
print("=" * 78)

c_all = pd.concat([results[n]["contrasts"] for n in CONFIGS], ignore_index=True)
c_sum = []
for label in c_all["Contrast"].unique():
    sub = c_all[c_all["Contrast"] == label]
    c_sum.append({
        "Contrast": label,
        **{f"Diff[{r.Config}]": round(r.Difference, 5) for r in sub.itertuples()},
        "Same_sign_all": bool(len(set(np.sign(sub["Difference"]))) == 1),
        "AdjCI_excludes_0_all": bool(sub["AdjCI_Excludes_0"].all()),
    })
print("\nCorrected-cosine contrasts:")
display(pd.DataFrame(c_sum))

m_all = pd.concat([results[n]["margin"] for n in CONFIGS], ignore_index=True)
m_sum = []
for grp in m_all["Incorrect_Group"].unique():
    sub = m_all[m_all["Incorrect_Group"] == grp]
    m_sum.append({
        "Incorrect_Group": grp,
        **{f"Median[{r.Config}]": round(r.Median_Margin, 5) for r in sub.itertuples()},
        **{f"Pct_closer_opp[{r.Config}]": round(r.Pct_Closer_to_Opposite, 1)
           for r in sub.itertuples()},
        "Median_CI_excludes_0_all_positive":
            bool((sub["Median_Margin_CI_Lower"] > 0).all()),
        "PctCloser_CI_above_50_all":
            bool((sub["Pct_Closer_to_Opposite_CI_Lower"] > 50).all()),
    })
print("\nMargin analysis:")
display(pd.DataFrame(m_sum))

print(
    "\nREPORTING GUIDE\n"
    "- Claim 'incorrect tweets resemble the opposite class' ONLY if, in every\n"
    "  configuration, the median-margin CI is above 0 AND the %-closer CI is\n"
    "  above 50%. A positive MEAN with a ~0 median is a skewed tail, not a\n"
    "  group-wide effect.\n"
    "- A result present in 'main' but absent in 'masked_artifacts' is a topic\n"
    "  artifact (e.g. mkr), not class resemblance.\n"
    "- All intervals are conditional on classifier-selected groups: call them\n"
    "  'bootstrap intervals', not significance tests. No p-values are reported\n"
    "  because the permutation null (exchangeability) is violated by selection.\n"
    "- Absolute cosine values depend on the vectorizer; interpret contrasts.\n"
    "- Independent validation: repeat section 2 with sentence embeddings."
)

Python 3.13.16 | numpy 2.1.3 | pandas 2.2.3 | sklearn 1.6.1
N_BOOT=2000, SEED=42, ALPHA=0.05

0. AUDIT (raw groups, before de-duplication)


,Group,Documents,Unique_Texts,Duplicate_%,Contains_rt_%,Contains_mkr_%
0,Inc Not-CB,1432,1427,0.349,9.148,0.209
1,Inc Other-CB,1671,1663,0.479,8.857,7.062
2,Cor Not-CB,3185,3154,0.973,11.429,38.493
3,Cor Other-CB,2163,2141,1.017,14.840,0.046


Texts appearing in more than one group (label inconsistency candidates): 12


,Original_N,Duplicates_removed,After_dedup_N
persistent_group,,,
Persistently incorrect Not-CB,1432,5,1427
Persistently incorrect Other-CB,1671,8,1663
Persistently correct Not-CB,3185,31,3154
Persistently correct Other-CB,2163,22,2141



CONFIG: main  {'ngram_range': (1, 2), 'min_df': 3, 'max_df': 0.95, 'max_features': 5000, 'mask': False}
  Vocabulary: 4630 | matrix: (8323, 4630)
  Empty vectors dropped by group:


...


5. ROBUSTNESS SUMMARY

Corrected-cosine contrasts:


,Contrast,Diff[main],Diff[masked_artifacts],Diff[alt_unigram],Same_sign_all,AdjCI_excludes_0_all
0,Inc Not-CB closer to Cor Other-CB than to Cor ...,0.38504,0.18776,0.3883,True,True
1,Inc Other-CB closer to Cor Not-CB than to Cor ...,0.06114,0.20633,0.0558,True,True



Margin analysis:


,Incorrect_Group,Median[main],Median[masked_artifacts],Median[alt_unigram],Pct_closer_opp[main],Pct_closer_opp[masked_artifacts],Pct_closer_opp[alt_unigram],Median_CI_excludes_0_all_positive,PctCloser_CI_above_50_all
0,Inc Not-CB,0.02545,0.00923,0.02560,90.7,68.2,90.2,True,True
1,Inc Other-CB,-0.00406,0.00857,-0.00422,39.6,70.0,39.1,False,False



REPORTING GUIDE
- Claim 'incorrect tweets resemble the opposite class' ONLY if, in every
  configuration, the median-margin CI is above 0 AND the %-closer CI is
  above 50%. A positive MEAN with a ~0 median is a skewed tail, not a
  group-wide effect.
- A result present in 'main' but absent in 'masked_artifacts' is a topic
  artifact (e.g. mkr), not class resemblance.
- All intervals are conditional on classifier-selected groups: call them
  'bootstrap intervals', not significance tests. No p-values are reported
  because the permutation null (exchangeability) is violated by selection.
- Absolute cosine values depend on the vectorizer; interpret contrasts.
- Independent validation: repeat section 2 with sentence embeddings.


In [24]:
t = persistent_four_df[persistent_four_df["lemmatized_tweet"].str.strip() != ""]
n_grp = t.groupby("lemmatized_tweet")["persistent_group"].nunique()
shared = t[t["lemmatized_tweet"].isin(n_grp[n_grp > 1].index)]
display(shared.sort_values("lemmatized_tweet")[["row_id","tweet_text", "lemmatized_tweet", "true_label", "persistent_group"]])

,row_id,tweet_text,lemmatized_tweet,true_label,persistent_group
24034,24034,@emmaxxxchloe is a bully :(,bully,other_cyberbullying,Persistently correct Other-CB
5383,5383,@polzie @moshanthi who should we bully?,bully,not_cyberbullying,Persistently incorrect Not-CB
22831,22831,@voretaq7 you don't know that.,dont know,other_cyberbullying,Persistently incorrect Other-CB
2583,2583,@ravavilad_film dont' we know it!,dont know,not_cyberbullying,Persistently incorrect Not-CB
24570,24570,@archivistgeek :face_blowing_a_kiss:,faceblowingakiss,other_cyberbullying,Persistently correct Other-CB
24591,24591,@arcbloodnorcal will do! :face_blowing_a_kiss:,faceblowingakiss,other_cyberbullying,Persistently incorrect Other-CB
23450,23450,:face_blowing_a_kiss: http://t.co/2qgnwhp3de,faceblowingakiss,other_cyberbullying,Persistently correct Other-CB
685,685,i just found these. http://t.co/lyam5snq1c,found,not_cyberbullying,Persistently incorrect Not-CB
22407,22407,found them,found,other_cyberbullying,Persistently incorrect Other-CB
23912,23912,= a fucking movie! ð¥,fucking movie,other_cyberbullying,Persistently correct Other-CB
